# PyMuPDF

In [4]:
import pymupdf

In [5]:
filename = "input_dict_files/rodegem_rundi1970_o.pdf"

rundi_dict_pdf = pymupdf.open(filename)  # or pymupdf.Document(filename)

In [6]:
rundi_sample_page = rundi_dict_pdf.load_page(27)

In [7]:
rundi_sample_page.get_text().split('\n')

['A',
 'a, (inv.), interjection de négation, de dénéga\xad',
 'tion : non. Peut être prononcée à bouche entrou\xad',
 'verte ou à bouche fermée. Dans ce dernier cas, ',
 'elle est souvent redoublée en a a, le second ',
 'étant prononcé légèrement plus haut.',
 'umwaba, imy-, 3|4, surface de terre arable dans ',
 'le marais. Mw-irima ry’imyaba, durant la cul\xad',
 'ture des marais.',
 'akâba, utw-, 12|13, lopin de terre arable. Drain ',
 "dans les champs de marais. Akâba n'umugazo ",
 'barekérwa mu busïze bw’îmfûnzo, dans le ter\xad',
 'rain défriché, parmi les papyrus on dégage un ',
 'drain appelé akâba.',
 'kwâba, -âvye, crier vers; avoir recours à, ',
 'recourir à. Hàgira ikibâye gishikiye umuryango ',
 'bàca bâgenda kwâba umupfùmu, si un malheur ',
 'frappait la famille, ils allaient aussitôt consulter ',
 'le devin.',
 'kwabàba, -vye, (-ab--), tendre la main pour ',
 'prendre en se dressant sur la pointe des pieds. ',
 'Il Commencer à ramper (enfant). Syn. kwavüra. ',
 '|( ',
 'E

In [12]:
import re

def insert_start_end_tokens(text: str) -> str:
    """
    Inserts '<start>' before each dictionary entry.

    Assumes each entry starts at a new line and begins with 2
    comma-separated fields, with the second .
    """

    # Step 1: insert <start>
    start_pattern = re.compile(
        r'^((?:[^,\n]+,\s*){1,2}\d+(?:\|\d+)?)',
        re.MULTILINE
    )
    text = start_pattern.sub(r'<start>\1', text)

    # # Step 2: for each <start>, find first '.' and insert <end>
    # def add_end_token(match):
    #     segment = match.group(0)
    #     dot_index = segment.find('.')
    #     if dot_index != -1:
    #         return segment[:dot_index + 1] + '<end>' + segment[dot_index + 1:]
    #     return segment  # no full stop found

    # Match from <start> up to next <start> or end of text
    entry_pattern = re.compile(r'<start>.*?(?=<start>|$)', re.DOTALL)
    # text = entry_pattern.sub(add_end_token, text)

    return text

In [16]:
# result = insert_start_end_tokens(rundi_sample_page.get_text())
# insert_start_end_tokens(rundi_sample_page.get_text())

"A\na, (inv.), interjection de négation, de dénéga\xad\ntion : non. Peut être prononcée à bouche entrou\xad\nverte ou à bouche fermée. Dans ce dernier cas, \nelle est souvent redoublée en a a, le second \nétant prononcé légèrement plus haut.\n<start>umwaba, imy-, 3|4, surface de terre arable dans \nle marais. Mw-irima ry’imyaba, durant la cul\xad\nture des marais.\n<start>akâba, utw-, 12|13, lopin de terre arable. Drain \ndans les champs de marais. Akâba n'umugazo \nbarekérwa mu busïze bw’îmfûnzo, dans le ter\xad\nrain défriché, parmi les papyrus on dégage un \ndrain appelé akâba.\nkwâba, -âvye, crier vers; avoir recours à, \nrecourir à. Hàgira ikibâye gishikiye umuryango \nbàca bâgenda kwâba umupfùmu, si un malheur \nfrappait la famille, ils allaient aussitôt consulter \nle devin.\nkwabàba, -vye, (-ab--), tendre la main pour \nprendre en se dressant sur la pointe des pieds. \nIl Commencer à ramper (enfant). Syn. kwavüra. \n|( \nEtre presque \négal, approcher \nde prés. \nUwàbâba umukù

In [17]:
# for entry in result.split('<start>'):
#     print(entry)
#     print('\n')

A
a, (inv.), interjection de négation, de dénéga­
tion : non. Peut être prononcée à bouche entrou­
verte ou à bouche fermée. Dans ce dernier cas, 
elle est souvent redoublée en a a, le second 
étant prononcé légèrement plus haut.



umwaba, imy-, 3|4, surface de terre arable dans 
le marais. Mw-irima ry’imyaba, durant la cul­
ture des marais.



akâba, utw-, 12|13, lopin de terre arable. Drain 
dans les champs de marais. Akâba n'umugazo 
barekérwa mu busïze bw’îmfûnzo, dans le ter­
rain défriché, parmi les papyrus on dégage un 
drain appelé akâba.
kwâba, -âvye, crier vers; avoir recours à, 
recourir à. Hàgira ikibâye gishikiye umuryango 
bàca bâgenda kwâba umupfùmu, si un malheur 
frappait la famille, ils allaient aussitôt consulter 
le devin.
kwabàba, -vye, (-ab--), tendre la main pour 
prendre en se dressant sur la pointe des pieds. 
Il Commencer à ramper (enfant). Syn. kwavüra. 
|( 
Etre presque 
égal, approcher 
de prés. 
Uwàbâba umukùru, vice-président, lieutenant. 
Aramwabâba mu 

# Tesseract

In [1]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image, ImageOps, ImageFilter
import time

In [2]:
def ocr_pdf_pipeline(pdf_path, image_dpi=300, languages ="fra+latn", ocr_config = '--oem 3 --psm 3' ):
    """
    Extracts text page-by-page to keep memory usage low.
    """
    full_text = []

    # convert_from_path returns a list of PIL images. 
    # To be truly memory efficient with large PDFs, we process them one by one.
    try:
        # We use a generator-like approach or process in small chunks
        # thread_count can speed up conversion, but increases CPU/RAM usage
        images = convert_from_path(pdf_path, dpi=image_dpi) 

        for i, image in enumerate(images):
            print(f"Processing page {i + 1}...")
            
            # 1. Perform OCR on the single page image
            
            # Specify languages: e.g., 'fra+yor' or 'fra+latn'
            # 'fra' handles the French vocabulary/grammar
            # 'latn' handles Latin characters/accents not specific to French

            # --- CROP LOGIC ---
            width, height = image.size
            
            crop_box = (
                width * 0.05,  # 5% from left
                height * 0.05, # 5% from top
                width * 0.95,  # 95% across (5% from right)
                height * 0.95  # 95% down (5% from bottom)
            )
            
            cropped_image = image.crop(crop_box)
           
            # if you wanna see the images
            cropped_image.save(f'sample_page_{i}.jpg')

            # Execute OCR
            text = pytesseract.image_to_string(cropped_image, lang=languages, config=ocr_config)
        
            # 2. Append text to our list
            full_text.append(text)
            
            # 3. Explicitly clear the image from memory
            cropped_image.close()
            image.close()     
            break
        return "\n\n".join(full_text)

    except Exception as e:
        return f"An error occurred: {e}"

In [33]:
# import os
# os.environ['TESSDATA_PREFIX'] = '/Users/harshamarsha/Documents/CourseWork/Bantu_HiWi/digitizing/tessdata_best/'

In [21]:
filename = "input_dict_files/rundi_sample.pdf"

start_time = time.perf_counter()
extracted_content = ocr_pdf_pipeline(
    
    filename, 
                                     
    image_dpi=900, 
                                     
    languages = "script/Latin", 
                                     
    ocr_config = '--oem 3 --psm 3 -c load_system_dawg=0 -c load_freq_dawg=0'
)

end_time = time.perf_counter()

print(f'Time taken to OCR: {(end_time-start_time):.2f} seconds')

Processing page 1...
Time taken to OCR: 18.21 seconds


In [22]:
print(extracted_content)

a, (inv.), interjection de négation, de dénéga-
tion: non. Peut être prononcée à bouche entrou-
verte ou à bouche fermée. Dans ce dernier cas,
eke est souvent redoublée en a a, le second
étant prononcé légèrement plus haut.

umwaba, imy-, 34, surface de terre arable dans
le marais, Mw-irima ryimyvaba, durant ta cul-
fure des maårais.

akāba, utw-, 12|13, lopin de terre arable. Drain
dans les champs de marais. Akaba n'umugazo
barekérwa mu busříze bw'îmfůnzo, dans le ter-
rain défriché, parmi les papyrus on dégage un
drain appelé akāba.

kwâba, -âvye, crier vers; avoir recours ğ,
recourir à. Hšgira ikibâye gishikíye umuryango
băca bågenda kwâôba umupfůmu, si un malheur
frappait la famille, ils allaient aussitôt consulier
le devin.

kwabāäba, -vye, (-ab--), tendre la main pour
prendre en se dressant sur la pointe des pieds.
li Commencer à ramper (enfant). Syn. kwavūra.
| Etre presque égal, approcher de près.
Uwăbāba umukůru, vice-président, lieutenant.
Aramwabāba mu kuvůka, is ont presque

# Tesseract + Line segmentation

In [1]:
import fitz  # PyMuPDF
import time
import os
import cv2
import numpy as np
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path, pdfinfo_from_path
from PIL import Image, ImageFilter
from tqdm.notebook import tqdm
import gc # Garbage Collector


# Set Tesseract to use multiple threads internally
os.environ["OMP_THREAD_LIMIT"] = "8"

In [17]:
# def save_entry_to_file(entry_str, file_path="dictionary_output.txt"):
#     """Appends a single entry to the text file immediately."""
#     with open(file_path, "a", encoding="utf-8") as f:
#         f.write(entry_str + "\n\n")
        

# def find_footer_separator_cv2(pil_img):
#     # 1. Convert to OpenCV grayscale
#     img = np.array(pil_img.convert('L'))
#     h, w = img.shape
    
#     # 2. Focus on the bottom 30% and binarize
#     # We use a higher threshold (220) to catch very faint/thin lines
#     start_y = int(h * 0.50)
#     roi = img[start_y:, :]
#     _, binary = cv2.threshold(roi, 220, 255, cv2.THRESH_BINARY_INV)
    
#     # 3. Create a horizontal "needle" kernel
#     # This kernel is 1 pixel tall but 50 pixels wide.
#     # It acts like a comb that only lets horizontal structures through.
#     horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
    
#     # 4. Apply Morphology (Erosion followed by Dilation)
#     # This deletes everything that isn't at least 50 pixels wide horizontally.
#     detected_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    
#     # 5. Find the Y-coordinate of the remaining pixels
#     # We look for the row with the most white pixels in the 'detected_lines' mask
#     row_sums = np.sum(detected_lines, axis=1)
    
#     if np.max(row_sums) > 0:
#         # Find the index of the row with the maximum "line-ness"
#         line_relative_y = np.argmax(row_sums)
#         global_y = start_y + line_relative_y
        
#         # Buffer: return 15 pixels above the line to ensure it's fully gone
#         return global_y - 15
        
#     return h

    
# def ocr_pdf_lineseg_pipeline(pdf_path, image_dpi=600, languages="fra+latn", ocr_config='--oem 3 --psm 3'):
    
#     output_file = "ocr_results.txt"
#     # Clear file if it exists from a previous run
#     if os.path.exists(output_file):
#         os.remove(output_file)
    
#     # Get page count without loading images
#     info = pdfinfo_from_path(pdf_path)
#     max_pages = info["Pages"]
    
#     for page_num in tqdm(range(1, max_pages + 1), desc="OCR Progress", unit="page"): 
        
#         print(f"Processing page {page_num}/{max_pages}...")
        
#         images = convert_from_path(pdf_path, dpi=image_dpi, first_page=page_num, last_page=page_num)
#         if not images:
#             continue
            
#         image = images[0]

#         # 1. Detect and Crop Footer
#         footer_y = find_footer_separator_cv2(image)
#         if footer_y < image.size[1]:
#             image = image.crop((0, 0, image.size[0], footer_y))   
                
#         width, height = image.size
#         left_m, right_m = width * 0.05, width * 0.95
#         top_m, mid = height * 0.05, width / 2
        
#         columns = [
#             ("Left", image.crop((left_m, top_m, mid, height))),
#             ("Right", image.crop((mid, top_m, right_m, height)))
#         ]

#         for col_name, col_img in columns:
#             # debug image
#             col_img.save(f'p{page_num}_{col_name}.png')
            
#             data = pytesseract.image_to_data(col_img, lang=languages, config=ocr_config, output_type=Output.DICT)
            
#             # Step 1: Reconstruct lines
#             lines = []
#             n_words = len(data['text'])
#             current_line_text, current_line_x = [], None

#             for j in range(n_words):
#                 if int(data['conf'][j]) > 0:
#                     if int(data['word_num'][j]) == 1:
#                         if current_line_text:
#                             lines.append({'x': current_line_x, 'text': " ".join(current_line_text)})
#                         current_line_x = data['left'][j]
#                         current_line_text = [data['text'][j]]
#                     else:
#                         current_line_text.append(data['text'][j])
            
#             if current_line_text:
#                 lines.append({'x': current_line_x, 'text': " ".join(current_line_text)})

#             if not lines:
#                 continue

#             # Step 2: Thresholding
#             median_x = np.median([l['x'] for l in lines])
#             indent_threshold = 40 # Adjust to 90 if 40 is too sensitive for 600dpi

#             # Step 3: Group and Write to File
#             current_entry = []
#             for line in lines:
#                 is_indent = line['x'] > (median_x + indent_threshold)

#                 if is_indent:
#                     if current_entry:
#                         # --- SAVE POSITION 1: End of an entry detected by a new indent ---
#                         entry_str = " ".join(current_entry).strip()
#                         save_entry_to_file(entry_str, output_file)
                    
#                     current_entry = [line['text']]
#                 else:
#                     if current_entry:
#                         current_entry.append(line['text'])
#                     else:
#                         current_entry = [line['text']]

#             if current_entry:
#                 # --- SAVE POSITION 2: Final entry of the column ---
#                 entry_str = " ".join(current_entry).strip()
#                 save_entry_to_file(entry_str, output_file)

#         # Cleanup Memory
#         for _, c_img in columns:
#             c_img.close()
#         image.close()
#         del images 
#         if page_num == 4: break
#     print(f"Finished! All entries saved to {output_file}")
#     return True

In [241]:
# --- Helper Functions + Main OCR Function---
def save_entry_to_file(entry_str, file_path="ocr_results.txt"):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(entry_str + "\n\n")
def find_footer_separator_cv2(pil_img):
    # 1. Convert to OpenCV grayscale
    img = np.array(pil_img.convert('L'))
    h, w = img.shape
    
    # 2. Focus on the bottom 70% and binarize
    # We use a higher threshold (220) to catch very faint/thin lines
    start_y = int(h * 0.30)
    roi = img[start_y:, :]
    _, binary = cv2.threshold(roi, 220, 255, cv2.THRESH_BINARY_INV)
    
    # 3. Create a horizontal "needle" kernel
    # This kernel is 1 pixel tall but 100 pixels wide.
    # It acts like a comb that only lets horizontal structures through.
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (100, 1))
    
    # 4. Apply Morphology (Erosion followed by Dilation)
    # This deletes everything that isn't at least 100 pixels wide horizontally.
    detected_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    
    # 5. Find the Y-coordinate of the remaining pixels
    # We look for the row with the most white pixels in the 'detected_lines' mask
    row_sums = np.sum(detected_lines, axis=1)
    
    if np.max(row_sums) > 0:
        # Find the index of the row with the maximum "line-ness"
        line_relative_y = np.argmax(row_sums)
        global_y = start_y + line_relative_y
        
        return global_y - 10
        
    return h

def find_dynamic_split_point(pil_img):
    # to divide a page into 2 by identifying the divider between the columns.
    
    # 1. Convert to grayscale and binarize
    bw_arr = np.array(pil_img.convert('L'))
    # 0 = black (text), 1 = white (background)
    bw_arr = (bw_arr > 128).astype(np.uint8) 
    
    h, w = bw_arr.shape
    
    # 2. Focus on the middle 20% of the page width
    # This ensures we don't accidentally split on a margin
    search_start = int(w * 0.35)
    search_end = int(w * 0.65)
    mid_zone = bw_arr[:, search_start:search_end]
    
    # 3. Vertical Projection: Sum the white pixels in each column
    # The gutter will be the column with the MOST white pixels
    column_white_sums = np.sum(mid_zone, axis=0)
    
    # Find the index of the column that is "whitest"
    best_gutter_relative = np.argmax(column_white_sums)
    
    # 4. Return the absolute X coordinate
    dynamic_mid = search_start + best_gutter_relative
    
    # Debug: if the "whitest" column isn't significantly white, fallback to w/2
    if column_white_sums[best_gutter_relative] < (h * 0.95):
        return int(w / 2)
        
    return dynamic_mid

def ocr_pdf_lineseg_pipeline(pdf_path, image_dpi=600, languages="fra+latn",
                            ocr_config='--oem 3 --psm 3',
                            output_file="ocr_results.txt",
                            export_pics=False, limit_max_p=None):

    max_pages = pdfinfo_from_path(pdf_path)["Pages"]

    # helper: write buffered entry
    def write_entry(entry, marker=None):
        if not entry: return
        s = " ".join(entry).strip()
        if s.endswith(('-', '–', '—')):
            s = s[:-1].strip()
        with open(output_file, "a", encoding="utf-8") as f:
            f.write(f"{s}\n{marker}\n\n" if marker else f"{s}\n\n")

    open(output_file, "w", encoding="utf-8").close() # create fresh file

    current_entry, held_marker = [], None

    for page_num in tqdm(range(1, max_pages + 1), desc="Processing Dictionary", unit="page"):
        try:
            images = convert_from_path(pdf_path, dpi=image_dpi, first_page=page_num, last_page=page_num)
            if not images: continue
            img = images[0]

            # crop footer + split columns
            footer_y = find_footer_separator_cv2(img)
            if footer_y < img.size[1]:
                img = img.crop((0, 0, img.size[0], footer_y))

            w, h = img.size
            mid = find_dynamic_split_point(img)
            columns = [
                img.crop((w * 0.05, h * 0.05, mid * 1.02, h)),
                img.crop((mid, h * 0.05, w * 0.95, h))
            ]

            for i, col_img in enumerate(columns):

                if export_pics:
                    col_img.save(f'p{page_num}_{i+1}.png')

                data = pytesseract.image_to_data(
                    col_img, lang=languages, config=ocr_config, output_type=Output.DICT)
                col_img.close()

                # reconstruct lines from OCR words
                lines, curr_text, curr_x = [], [], None
                for j in range(len(data['text'])):
                    if int(data['conf'][j]) > 0:
                        if int(data['word_num'][j]) == 1:
                            if curr_text:
                                lines.append({'x': curr_x, 'text': " ".join(curr_text)})
                            curr_x, curr_text = data['left'][j], [data['text'][j]]
                        else:
                            curr_text.append(data['text'][j])
                if curr_text:
                    lines.append({'x': curr_x, 'text': " ".join(curr_text)})
                if not lines: continue

                # extract guide word
                first = lines[0]['text'].strip()
                m = re.match(r'^([A-Z]{3})(?:\s+|$)', first)
                guide = m.group(1) if m else ""
                if guide:
                    lines[0]['text'] = first[len(guide):].strip()
                if not lines[0]['text']:
                    lines.pop(0)
                if not lines: continue

                marker = f"COLUMN-START-P{page_num}-C{i+1}"
                marker = f"{marker}-{guide}" if guide else marker

                median_x = np.median([l['x'] for l in lines])
                first_line = lines[0]
                tokens = first_line['text'].split()

                is_new = (
                    first_line['x'] > (median_x + 40) and
                    tokens and tokens[0].endswith(',')
                )

                # hyphen carry across columns
                hyphen_carry = current_entry and current_entry[-1].strip().endswith(('-', '–', '—'))

                if hyphen_carry:
                    current_entry[-1] = current_entry[-1].strip()[:-1] + first_line['text'].strip()
                    held_marker, start_idx = marker, 1

                elif is_new:
                    write_entry(current_entry)
                    current_entry = [first_line['text'].strip()]
                    with open(output_file, "a", encoding="utf-8") as f:
                        f.write(f"{marker}\n\n")
                    held_marker, start_idx = None, 1

                else:
                    current_entry.append(first_line['text'].strip())
                    held_marker, start_idx = marker, 1

                # process remaining lines
                for line in lines[start_idx:]:
                    txt = line['text'].strip()
                    tokens = txt.split()
                    is_new = (
                        line['x'] > (median_x + 40) and
                        tokens and tokens[0].endswith(',')
                    )

                    if is_new:
                        write_entry(current_entry, held_marker)
                        current_entry, held_marker = [txt], None
                    else:
                        if current_entry and current_entry[-1].strip().endswith(('-', '–', '—')):
                            current_entry[-1] = current_entry[-1].strip()[:-1] + txt
                        else:
                            current_entry.append(txt)

            img.close()
            del images, img
            gc.collect()

        except Exception as e:
            print(f"Skipping page {page_num} due to error: {e}")
            continue

        if limit_max_p and page_num == limit_max_p:
            break

    write_entry(current_entry, held_marker)
    return f"Success! Results appended to {output_file}"

In [247]:
# filename = "input_dict_files/rundi_sample.pdf" # sample with 3 pages
filename = "input_dict_files/rundi_working_file.pdf" # main

start_time = time.perf_counter()
extracted_content = ocr_pdf_lineseg_pipeline(
    
    filename, 
                                     
    image_dpi=600, 
                                     
    languages = "script/Latin", 
                                     
    ocr_config = f"--oem 3 --psm 3 -c load_system_dawg=0 -c load_freq_dawg=0",

    output_file = "rundi_extracted_full.txt",

    export_pics = False, # debug

    limit_max_p = None # debug
)

end_time = time.perf_counter() 

print(f'Time taken to OCR: {(end_time-start_time)/60:.2f} min')
extracted_content

Processing Dictionary:   0%|          | 0/588 [00:00<?, ?page/s]

Skipping page 306 due to error: '<' not supported between instances of 'int' and 'NoneType'
Skipping page 524 due to error: '<' not supported between instances of 'int' and 'NoneType'
Time taken to OCR: 80.92 min


'Success! Results appended to rundi_extracted_full.txt'

# Text Cleaning

In [178]:
print(sorted(__import__('collections').Counter(open('rundi_extracted_cleaned_p30.txt', encoding='utf-8').read()).items(), key=lambda x: x[1]))


[('Ê', 1), ('ş', 1), ('Œ', 1), ('ť', 1), ('Ū', 1), ('Ì', 1), ('ż', 1), ('Ó', 1), ('Â', 1), ('ž', 2), ('İ', 2), ('ğ', 2), ('Ī', 3), ('ď', 3), ('É', 3), ('À', 4), ('X', 5), ('ć', 6), ('Z', 6), ('ň', 8), ('Í', 9), ('ì', 9), ('ŭ', 10), ('Q', 10), ('—', 10), ('œ', 11), ('Î', 11), ('æ', 12), ('J', 17), ('«', 18), ('»', 18), ('ò', 21), ('ï', 25), ('ě', 25), ('’', 30), ('W', 33), ('ĉ', 33), ('ç', 34), ('Y', 35), ('?', 38), ('ë', 42), ('D', 43), ('ė', 45), ('H', 63), (':', 65), ('L', 65), ('ē', 65), ('O', 80), ('V', 81), ('G', 82), ('ō', 84), ('!', 86), ('M', 89), ('ù', 90), ('T', 92), ('C', 99), ('ô', 102), ('F', 106), ('í', 108), ('R', 115), ('ū', 123), ('ī', 126), ('ú', 140), ('N', 143), ('B', 146), ('E', 147), ('I', 148), ('û', 151), (';', 159), ('ó', 174), ('î', 175), ('U', 180), ('A', 203), ('9', 249), ('6', 251), ('P', 253), ('8', 254), ('0', 263), ('è', 274), ('5', 274), ('7', 280), ('K', 283), ('x', 292), ('ê', 350), ('ă', 376), ('4', 384), ('2', 405), ('à', 427), ('3', 432), ('ā', 442

In [252]:
# ! IMP --- Create a string set of allowed characters
unique_chars = set(open("rundi_extracted_cleaned_full.txt", "r", encoding="utf-8").read())

# Filter out non-printable/control chars and join with their uppercase forms
whitelist = "".join(sorted({c for c in unique_chars if c.strip()} | {c.upper() for c in unique_chars if c.strip()}))
whitelist+=' '

print(f"{whitelist}")

# with open('rundi_whitelist_char_raw.txt', 'w', encoding='utf-8') as file:
#     file.write(whitelist)

!'(),-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz|«»ÀÁÂÆÇÈÉÊËÌÍÎÏÒÓÔÙÚÛÝàáâæçèéêëìíîïòóôùúûýĀāĂăĆćĈĉĎďĒēĖėĚěĞğĠġĢģĤĥĪīİĴĵŃńŇňŌōŒœŚśŜŝŞşŢŤťŪūŬŭŹźŻżŽž—’ 


In [35]:
import re
import unicodedata

# 0. Load the whitelist once
try:
    with open('rundi_whitelist_char_baseline.txt', 'r', encoding='utf-8') as f:
        whitelist_raw = f.read()
        # Include common structural characters just in case they aren't in your text file
        SAFE_CHARS = set(unicodedata.normalize('NFD', whitelist_raw)) | set(" \n\r\t().,:-")
except FileNotFoundError:
    SAFE_CHARS = set()
    print("Warning: rundi_whitelist_char_baseline.txt not found.")

def clean_rundi_dict(text):
    
    # 1. Global character-level swaps (Pre-decomposition for clarity)
    base_map = {
        # Diacritic Swaps
        'å': 'â', 'ä': 'a', 'ů': 'û', 'ö': 'ō', 'ẹ': 'e', 'ř': 'i','ü':'u','ņ':'n','ċ':'c','ķ':'k','ĝ':'g',
        'ľ':'l', 'š':'','č':'c',
        
        # Stroke/Special Character Base-Mapping
        'đ': 'd', 'ļ': 'l', 'ł': 'l', 'ħ': 'h', 'ø': 'o', 'ŧ': 't',
    }
    
    # 2. Expand to UPPERCASE
    cleanup_map = {
        **base_map, 
        **{k.upper(): v.upper() for k, v in base_map.items()}
    }
    
    # run cleanup on characters
    for search, replace in cleanup_map.items():
        text = text.replace(search, replace)

    
    # 2. Bracket normalization
    text = re.sub(r'[\[\{]', '(', text)
    text = re.sub(r'[\]\}]', ')', text)
    text = re.sub(r'\(+', '(', text)
    text = re.sub(r'\)+', ')', text)

    # 3. Decompose characters
    nfd_text = unicodedata.normalize('NFD', text)
    
    # 4. Define the Diacritic Mapping (Combining Marks)
    DIACRITIC_MAP = {
        '\u030a': '\u0302', # Ring -> Circumflex
        '\u0328': '',       # Ogonek -> Remove
        '\u030b': '\u0301', # Double Acute -> Acute
    }

    # 5. Filter/Map the decomposed string
    cleaned_chars = []
    for char in nfd_text:
        # Check if it is a diacritic (Mark, Nonspacing)
        is_diacritic = unicodedata.category(char) == 'Mn'

        if char in SAFE_CHARS:
            cleaned_chars.append(char)
        elif is_diacritic:
            if char in DIACRITIC_MAP:
                mapped_mark = DIACRITIC_MAP[char]
                if mapped_mark: # Only append if the map doesn't lead to removal
                    cleaned_chars.append(mapped_mark)
            else:
                # Rule: If diacritic not in SAFE or MAP, delete it (do nothing)
                pass
        else:
            # Rule: If base character is not in SAFE_CHARS, skip it
            # This keeps your text strictly within your known character set
            pass
    
    cleaned_nfd = "".join(cleaned_chars)

    # 6. Recompose back to standard characters (NFC)
    return unicodedata.normalize('NFC', cleaned_nfd)

In [36]:
rundi_auto_cleaned = clean_rundi_dict(open("rundi_extracted_full.txt").read())

with open('rundi_extracted_cleaned_full.txt', 'w', encoding='utf-8') as f:
    f.write(rundi_auto_cleaned)

In [46]:
import re

INPUT_FILE = "rundi_extracted_cleaned_full.txt"
OUTPUT_FILE = "rundi_extracted_cleaned_full.txt"

def normalize_column_spacing(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        content = f.read()
    # This replaces the entire "whitespace bridge" with exactly two newlines.
    fixed_content = re.sub(r'\s+(?=COLUMN-START)', r'\n\n', content)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(fixed_content)
    
    print(f"Cleanup complete! All COLUMN-START markers now have exactly two newlines.")

normalize_column_spacing(INPUT_FILE, OUTPUT_FILE)

Cleanup complete! All COLUMN-START markers now have exactly two newlines.


In [92]:
import re
import difflib
from collections import Counter

def extract_target_features(text):
    # common + OCR-misread diacritic characters in our code:
    char_pattern_raw = r"[!'(),-./0123456789:;?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[abcdefghijklmnopqrstuvwxyz{|}«»ÀÁÂÃÄÅÆÇÈÉÊËÍÎÏÒÓÔÖÙÚÛÜàáâãäåæçèéêëíîïòóôöùúûüĀāĂăĄąĆćĈĉĊċČčĐđĒēĖėĚěĦħĪīĶķĻļĽľŁłŅņŌōŒœŘřŠšŪūŬŭŮůŲųẸẹỌọ’„ ]+"

    # Combine patterns
    combined_regex = f"({char_pattern_raw})"
    matches = re.findall(combined_regex, text)
    return matches

def get_feature_diffs(raw_file, clean_file):
    # 1. Read files
    with open(raw_file, 'r', encoding='utf-8') as f:
        raw_tokens = f.read().split()
    with open(clean_file, 'r', encoding='utf-8') as f:
        clean_tokens = f.read().split()

    # 2. Match words (Tokens) to find where changes happened
    matcher = difflib.SequenceMatcher(None, raw_tokens, clean_tokens)
    edit_registry = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        # 'replace' means a word changed; 'delete'/'insert' handles dropped/added words
        if tag in ('replace', 'delete', 'insert'):
            
            # Combine the chunks being compared (handling unequal list lengths)
            r_chunk = "".join(raw_tokens[i1:i2])
            c_chunk = "".join(clean_tokens[j1:j2])

            # 3. Perform Character-Level Diff on the changed chunk
            char_matcher = difflib.SequenceMatcher(None, r_chunk, c_chunk)
            for c_tag, ci1, ci2, cj1, cj2 in char_matcher.get_opcodes():
                if c_tag != 'equal':
                    # Extract the specific character(s) that changed
                    from_chars = r_chunk[ci1:ci2]
                    to_chars = c_chunk[cj1:cj2]
                    
                    # Optional: Filter here if you only want to track vowel/bracket changes
                    if from_chars or to_chars:
                        edit_registry.append(f"{from_chars} -> {to_chars}")

    return Counter(edit_registry).most_common()

    

In [250]:
# AUTO CLEAN FUNCTION RESULT - DIFF
get_feature_diffs("rundi_extracted_full.txt", "rundi_extracted_cleaned_full.txt")

[('å -> â', 2092),
 ('ů -> û', 1740),
 ('} -> ', 1182),
 ('ö -> ō', 1035),
 ('ļ -> l', 742),
 ('} -> )', 716),
 ('ü -> u', 520),
 ('ä -> a', 460),
 ('č -> c', 459),
 ('ð -> ', 444),
 ('{ -> (', 408),
 ('/ -> ', 385),
 ('] -> )', 352),
 ('õ -> o', 315),
 ('ą -> a', 307),
 ('[ -> (', 295),
 ('ł -> l', 288),
 ('đ -> d', 241),
 ('ņ -> n', 220),
 ('ľ -> l', 151),
 ('+ -> ', 137),
 ('ẹ -> e', 124),
 ('ř -> i', 104),
 ('“ -> ', 85),
 ('„ -> ', 78),
 ('ħ -> h', 77),
 ('ã -> a', 74),
 ('` -> ', 68),
 ('ķ -> k', 67),
 ('ő -> ó', 60),
 ('į -> i', 60),
 ('ọ -> o', 56),
 ('= -> ', 55),
 ('ĝ -> g', 52),
 ('^ -> ', 52),
 ('š -> ', 50),
 ('_ -> ', 50),
 ('ı -> ', 50),
 ('ų -> u', 47),
 ('Į -> I', 46),
 ('& -> ', 42),
 ('ę -> e', 42),
 ('ą -> ', 37),
 (' -> a', 37),
 ('~ -> ', 35),
 ('čð -> c', 35),
 ('þ -> ', 26),
 ('ű -> ú', 26),
 ('‘ -> ', 26),
 ('@ -> ', 25),
 ('ļ] -> l)', 25),
 ('}) -> ', 24),
 ('Ķ -> K', 24),
 ('{ -> ', 23),
 ('ċ -> c', 19),
 ('\\ -> ', 18),
 ('° -> ', 18),
 ('ðč -> c', 17),
 ('ð

In [251]:
raw_charset = set(open("rundi_extracted_cleaned_full.txt", "r", encoding="utf-8").read())
clean_charset = set(open("rundi_whitelist_char_baseline.txt", "r", encoding="utf-8").read())

" ".join(char for char in raw_charset if char not in clean_charset)
# lists all the weird OCR characters after cleaning

'Ž İ \n ğ ť Ş Ţ ď ž Ď ý ĵ ň Ġ ġ Ğ ż ŝ Ì ş ś ź ģ ń ĥ'

# Ollama Prompting

In [267]:
# MLX Implementation of Ollama - Unified Memory

In [1]:
!export OLLAMA_MLX=1

In [2]:
import ollama
import pandas as pd
import json
import time
from tqdm.notebook import tqdm
import os

def fast_batch_process():
    # 1. Load and Parse with Truncation
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        raw_chunks = [e.strip() for e in f.read().split('\n\n') if len(e.strip()) > 3]

    processed_entries = []
    current_column = "UNKNOWN"
    
    # If START_COLUMN is set, start_processing is False until we find the marker
    start_processing = True if not START_COLUMN else False

    for chunk in raw_chunks:
        if chunk.upper().startswith("COLUMN-START"):
            current_column = chunk
            # Check if this marker matches our starting point
            if START_COLUMN and START_COLUMN in chunk:
                start_processing = True
                print(f"Found start marker: {chunk}. Beginning extraction...")
        else:
            # Only append if we have passed the start marker
            if start_processing:
                processed_entries.append((chunk, current_column))

    if not processed_entries:
        print("No entries found after the specified START_COLUMN. Check your marker string.")
        return

    # Initialize CSV
    with open(OUTPUT_CSV, 'w', encoding='utf-8') as f:
        f.write('')
        
    # 2. Process in Batches
    for i in tqdm(range(0, len(processed_entries), BATCH_SIZE), desc="Processing Batches", unit='batch'):
        batch_tuples = processed_entries[i : i + BATCH_SIZE]
        
        # ID is critical for the mapping logic
        batch_str = "\n".join([f"<entry id='{idx}'>{text}</entry>" for idx, (text, col) in enumerate(batch_tuples)])
        
        items = [] 
        for attempt in range(3): # Increased to 3 for better recovery
            try:
                response = ollama.chat(
                    model=MODEL,
                    messages=[
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user', 'content': f"Extract every entry within the tags into a JSON array labeled 'entries':\n{batch_str}"}
                    ],
                    format='json',
                    options={
                        'temperature': 0,
                        'num_ctx': 2048,
                        'num_predict': 1024,
                        'top_k': 1,
                        'top_p': 0.01
                    }
                )

                content = response['message']['content']
                batch_data = json.loads(content)
                
                # Normalize extracted list
                extracted_list = batch_data.get('entries', []) if isinstance(batch_data, dict) else batch_data
                if not isinstance(extracted_list, list): extracted_list = [extracted_list]

                # --- ROBUST MAPPING ---
                # Map by ID if available, otherwise fallback to index
                output_map = {str(item.get('id')): item for item in extracted_list if 'id' in item}
                
                batch_results = []
                for idx, (original_text, source_col) in enumerate(batch_tuples):
                    # Try matching by ID first
                    item = output_map.get(str(idx))
                    
                    # Fallback to positional index if IDs are missing but length matches
                    if item is None and len(extracted_list) == len(batch_tuples):
                        item = extracted_list[idx]
                    
                    if item:
                        # Clean the column marker
                        item['PAGE-COLUMN'] = "-".join(source_col.split("-")[2:4])
                        item['FULL_ENTRY'] = original_text
                        if 'id' in item: del item['id'] # Clean up internal ID
                        batch_results.append(item)
                
                items = batch_results
                break 
                
            except Exception as e:
                if attempt == 2:
                    print(f"Error at index {i}: {e}")
                else:
                    time.sleep(2)
                    continue

        # 3. Save Every Batch
        if items:
            pd.DataFrame(items).to_csv(
                OUTPUT_CSV, 
                mode='a', 
                index=False, 
                header=(i == 0), 
                encoding='utf-8'
            )
            
    print(f"\nExtraction complete. Results saved to {OUTPUT_CSV}")

In [3]:

# Run it
start_time = time.perf_counter()

# --- CONFIGURATION ---
# MODEL = "llama3.2:3b"
MODEL = "gemma4:e2b"
# MODEL = "gemma2:2b"
BATCH_SIZE = 4

INPUT_FILE = "rundi_extracted_cleaned_full.txt"

START_COLUMN = "P1-C1"  # functionality to start from selected page, default value: 'None' | "P28-C1"

OUTPUT_CSV = f"rundi_llm_tagged_{MODEL.split(":")[0]}_from{START_COLUMN}.csv"


SYSTEM_PROMPT = """
Extract Rundi dictionary data into a JSON array labeled "entries". 

For each <entry>, extract:
- WORD: Headword only.
- AFFIX: Prefix/suffix with a SINGLE hyphen (e.g., "-ye", "imy-"). 
- NOUN_CLASS: Numeric classes (e.g., "9|10", "11"). Skip if not found.
- DESCRIPTION: A concise French definition. Omit examples and long ethnographic notes. 
- POS: Predict from the DESCRIPTION (e.g., "v.", "n.", "adj.")

Rules:
1. Do not skip the POS tag!
2. DESCRIPTION must be a short gloss, not the entire entry.
3. Maintain the order of entries provided.
"""

fast_batch_process()

end_time = time.perf_counter()

print(f'Time taken to OCR: {(end_time-start_time)/60:.2f} min')


Found start marker: COLUMN-START-P1-C1. Beginning extraction...


Processing Batches:   0%|          | 0/4854 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [60]:
ERROR_LOG = """
Error at index 2528: Expecting value: line 1 column 1 (char 0)
Error at index 2544: Expecting value: line 1 column 1 (char 0)
Error at index 2592: Expecting value: line 148 column 19 (char 2668)
Error at index 2612: Expecting value: line 1 column 1 (char 0)
Error at index 2692: Expecting value: line 1 column 1 (char 0)
Error at index 2720: Expecting value: line 1 column 1 (char 0)
Error at index 2728: Expecting value: line 1 column 1 (char 0)
Error at index 2800: Expecting value: line 1 column 1 (char 0)
Error at index 2836: Expecting value: line 1 column 1 (char 0)
Error at index 2856: Expecting value: line 1 column 1 (char 0)
Error at index 2880: Expecting value: line 1 column 1 (char 0)
Error at index 2980: Expecting value: line 1 column 1 (char 0)
Error at index 3136: Expecting value: line 1 column 1 (char 0)
Error at index 3356: Expecting value: line 1 column 1 (char 0)
Error at index 3384: Expecting value: line 1 column 1 (char 0)
Error at index 3508: Expecting value: line 1 column 1 (char 0)
Error at index 3600: Expecting value: line 1 column 1 (char 0)
Error at index 3628: Expecting value: line 1 column 1 (char 0)
Error at index 3692: Expecting value: line 1 column 1 (char 0)
Error at index 3736: Expecting value: line 1 column 1 (char 0)
Error at index 3744: Expecting value: line 1 column 1 (char 0)
Error at index 3760: Expecting value: line 1 column 1 (char 0)
Error at index 3828: Expecting value: line 1 column 1 (char 0)
Error at index 3832: Expecting value: line 1 column 1 (char 0)
Error at index 3904: Expecting value: line 1 column 1 (char 0)
Error at index 3944: Expecting value: line 1 column 1 (char 0)
Error at index 3952: Expecting value: line 1 column 1 (char 0)
Error at index 3956: Expecting value: line 1 column 1 (char 0)
Error at index 3996: Expecting value: line 1 column 1 (char 0)
Error at index 4016: Expecting value: line 1 column 1 (char 0)
Error at index 4028: Expecting value: line 1 column 1 (char 0)
Error at index 4068: Expecting value: line 1 column 1 (char 0)
Error at index 4076: Expecting ',' delimiter: line 65 column 2 (char 1058)
Error at index 4132: Expecting value: line 1 column 1 (char 0)
Error at index 4176: Expecting value: line 1 column 1 (char 0)
Error at index 4204: Expecting value: line 1 column 1 (char 0)
Error at index 4300: Expecting value: line 1 column 1 (char 0)
Error at index 4312: Expecting value: line 1 column 1 (char 0)
Error at index 4328: Expecting value: line 1 column 1 (char 0)
Error at index 4348: Expecting value: line 1 column 1 (char 0)
Error at index 4372: Expecting value: line 1 column 1 (char 0)
Error at index 4428: Expecting value: line 1 column 1 (char 0)
Error at index 4488: Expecting value: line 1 column 1 (char 0)
Error at index 4504: Expecting value: line 1 column 1 (char 0)
Error at index 4576: Expecting value: line 1 column 1 (char 0)
Error at index 4592: Expecting value: line 1 column 1 (char 0)
Error at index 4760: Expecting value: line 1 column 1 (char 0)
Error at index 4852: Expecting value: line 1 column 1 (char 0)
Error at index 4876: Expecting value: line 1 column 1 (char 0)
Error at index 4944: Expecting value: line 1 column 1 (char 0)
Error at index 5160: Expecting value: line 1 column 1 (char 0)
Error at index 5176: Expecting value: line 1 column 1 (char 0)
Error at index 5208: Expecting value: line 1 column 1 (char 0)
Error at index 5344: Expecting value: line 1 column 1 (char 0)
Error at index 5420: Expecting value: line 1 column 1 (char 0)
Error at index 5464: Expecting value: line 1 column 1 (char 0)
Error at index 5476: Expecting value: line 1 column 1 (char 0)
Error at index 5492: Expecting value: line 1 column 1 (char 0)
Error at index 5512: Expecting value: line 1 column 1 (char 0)
Error at index 5520: Expecting value: line 1 column 1 (char 0)
Error at index 5576: Expecting value: line 1 column 1 (char 0)
Error at index 5608: Expecting value: line 1 column 1 (char 0)
Error at index 5612: Expecting value: line 1 column 1 (char 0)
Error at index 5628: Expecting value: line 1 column 1 (char 0)
Error at index 5664: Expecting value: line 1 column 1 (char 0)
Error at index 5724: Expecting value: line 1 column 1 (char 0)
Error at index 5744: Expecting value: line 1 column 1 (char 0)
Error at index 5884: Expecting value: line 1 column 1 (char 0)
Error at index 6060: Expecting value: line 1 column 1 (char 0)
Error at index 6112: Expecting value: line 1 column 1 (char 0)
Error at index 6128: Expecting value: line 1 column 1 (char 0)
Error at index 6176: Expecting value: line 1 column 1 (char 0)
Error at index 6256: Expecting value: line 1 column 1 (char 0)
Error at index 6272: Expecting value: line 1 column 1 (char 0)
Error at index 6320: Expecting value: line 1 column 1 (char 0)
Error at index 6344: Expecting value: line 1 column 1 (char 0)
Error at index 6400: Expecting value: line 1 column 1 (char 0)
Error at index 6432: Expecting value: line 1 column 1 (char 0)
Error at index 6512: Expecting value: line 1 column 1 (char 0)
Error at index 6532: Expecting value: line 1 column 1 (char 0)
Error at index 6540: Expecting value: line 1 column 1 (char 0)
Error at index 6564: Expecting value: line 1 column 1 (char 0)
Error at index 6572: Expecting value: line 1 column 1 (char 0)
Error at index 6588: Expecting value: line 1 column 1 (char 0)
Error at index 6632: Expecting value: line 1 column 1 (char 0)
Error at index 6728: Expecting value: line 1 column 1 (char 0)
Error at index 6732: Expecting value: line 1 column 1 (char 0)
Error at index 6748: Expecting value: line 1 column 1 (char 0)
Error at index 6856: Expecting value: line 1 column 1 (char 0)
Error at index 6880: Expecting value: line 1 column 1 (char 0)
Error at index 6884: Expecting value: line 1 column 1 (char 0)
Error at index 6904: Expecting value: line 1 column 1 (char 0)
Error at index 6924: Expecting value: line 1 column 1 (char 0)
Error at index 6984: Expecting value: line 1 column 1 (char 0)
Error at index 6992: Expecting value: line 1 column 1 (char 0)
Error at index 7020: Expecting value: line 1 column 1 (char 0)
Error at index 7048: Expecting value: line 1 column 1 (char 0)
Error at index 7084: Expecting value: line 1 column 1 (char 0)
Error at index 7100: Expecting value: line 1 column 1 (char 0)
Error at index 7176: Expecting value: line 1 column 1 (char 0)
Error at index 7192: Expecting value: line 1 column 1 (char 0)
Error at index 7244: Expecting value: line 1 column 1 (char 0)
Error at index 7260: Expecting ',' delimiter: line 130 column 1 (char 2429)
Error at index 7272: Expecting value: line 1 column 1 (char 0)
Error at index 7456: Expecting value: line 1 column 1 (char 0)
Error at index 7484: Expecting value: line 1 column 1 (char 0)
Error at index 7496: Expecting value: line 1 column 1 (char 0)
Error at index 7508: Expecting value: line 1 column 1 (char 0)
Error at index 7580: Expecting value: line 1 column 1 (char 0)
Error at index 7616: Expecting value: line 1 column 1 (char 0)
Error at index 7668: Expecting value: line 1 column 1 (char 0)
Error at index 7676: Expecting value: line 1 column 1 (char 0)
Error at index 7752: Expecting value: line 1 column 1 (char 0)
Error at index 7812: Expecting value: line 1 column 1 (char 0)
Error at index 7880: Expecting value: line 1 column 1 (char 0)
Error at index 7900: Expecting value: line 1 column 1 (char 0)
Error at index 7908: Expecting value: line 1 column 1 (char 0)
Error at index 7992: Expecting value: line 1 column 1 (char 0)
Error at index 7996: Expecting value: line 1 column 1 (char 0)
Error at index 8024: Expecting value: line 1 column 1 (char 0)
Error at index 8104: Expecting value: line 1 column 1 (char 0)
Error at index 8164: Expecting value: line 1 column 1 (char 0)
Error at index 8180: Expecting value: line 1 column 1 (char 0)
Error at index 8184: Expecting value: line 1 column 1 (char 0)
Error at index 8192: Expecting value: line 1 column 1 (char 0)
Error at index 8212: Expecting value: line 1 column 1 (char 0)
Error at index 8232: Expecting value: line 1 column 1 (char 0)
Error at index 8296: Expecting value: line 1 column 1 (char 0)
Error at index 8316: Expecting value: line 1 column 1 (char 0)
Error at index 8320: Expecting value: line 1 column 1 (char 0)
Error at index 8368: Expecting value: line 1 column 1 (char 0)
Error at index 8400: Expecting value: line 1 column 1 (char 0)
Error at index 8408: Expecting value: line 1 column 1 (char 0)
Error at index 8412: Expecting value: line 1 column 1 (char 0)
Error at index 8436: Expecting value: line 1 column 1 (char 0)
Error at index 8536: Expecting value: line 1 column 1 (char 0)
Error at index 8584: Expecting value: line 1 column 1 (char 0)
Error at index 8600: Expecting value: line 1 column 1 (char 0)
Error at index 8608: Expecting value: line 1 column 1 (char 0)
Error at index 8636: Expecting value: line 1 column 1 (char 0)
Error at index 8644: Expecting value: line 1 column 1 (char 0)
Error at index 8664: Expecting value: line 1 column 1 (char 0)
Error at index 8708: Expecting value: line 1 column 1 (char 0)
Error at index 8756: Expecting value: line 1 column 1 (char 0)
Error at index 8776: Expecting value: line 1 column 1 (char 0)
Error at index 8780: Expecting value: line 1 column 1 (char 0)
Error at index 8784: Expecting value: line 1 column 1 (char 0)
Error at index 8796: Expecting value: line 1 column 1 (char 0)
Error at index 8812: Expecting value: line 1 column 1 (char 0)
Error at index 8836: Expecting value: line 1 column 1 (char 0)
Error at index 8852: Expecting value: line 1 column 1 (char 0)
Error at index 8904: Expecting value: line 1 column 1 (char 0)
Error at index 8936: Expecting value: line 1 column 1 (char 0)
Error at index 8960: Expecting value: line 1 column 1 (char 0)
Error at index 8992: Expecting value: line 1 column 1 (char 0)
Error at index 9028: Expecting value: line 1 column 1 (char 0)
Error at index 9048: Expecting value: line 1 column 1 (char 0)
Error at index 9056: Expecting value: line 1 column 1 (char 0)
Error at index 9076: Expecting value: line 117 column 19 (char 2517)
Error at index 9156: Expecting value: line 1 column 1 (char 0)
Error at index 9224: Expecting value: line 1 column 1 (char 0)
Error at index 9260: Expecting value: line 1 column 1 (char 0)
Error at index 9272: Expecting value: line 1 column 1 (char 0)
Error at index 9280: Expecting value: line 1 column 1 (char 0)
Error at index 9288: Expecting value: line 1 column 1 (char 0)
Error at index 9312: Expecting value: line 1 column 1 (char 0)
Error at index 9424: Expecting value: line 1 column 1 (char 0)
Error at index 9464: Expecting value: line 1 column 1 (char 0)
Error at index 9472: Expecting value: line 1 column 1 (char 0)
Error at index 9476: Expecting value: line 1 column 1 (char 0)
Error at index 9496: Expecting value: line 1 column 1 (char 0)
Error at index 9516: Expecting value: line 1 column 1 (char 0)
Error at index 9520: Expecting value: line 1 column 1 (char 0)
Error at index 9576: Expecting value: line 1 column 1 (char 0)
Error at index 9700: Expecting value: line 1 column 1 (char 0)
Error at index 9732: Expecting value: line 1 column 1 (char 0)
Error at index 9752: Expecting value: line 1 column 1 (char 0)
Error at index 9760: Expecting value: line 1 column 1 (char 0)
Error at index 9800: Expecting value: line 1 column 1 (char 0)
Error at index 9864: Expecting value: line 1 column 1 (char 0)
Error at index 9876: Expecting value: line 1 column 1 (char 0)
Error at index 9936: Expecting value: line 1 column 1 (char 0)
Error at index 9960: Expecting value: line 1 column 1 (char 0)
Error at index 10008: Expecting value: line 1 column 1 (char 0)
Error at index 10076: Expecting value: line 1 column 1 (char 0)
Error at index 10144: Expecting value: line 1 column 1 (char 0)
Error at index 10168: Expecting value: line 1 column 1 (char 0)
Error at index 10184: Expecting value: line 1 column 1 (char 0)
Error at index 10252: Expecting value: line 1 column 1 (char 0)
Error at index 10324: Expecting value: line 1 column 1 (char 0)
Error at index 10328: Expecting value: line 1 column 1 (char 0)
Error at index 10356: Expecting value: line 1 column 1 (char 0)
Error at index 10396: Expecting ',' delimiter: line 126 column 2 (char 2049)
Error at index 10460: Expecting value: line 1 column 1 (char 0)
Error at index 10480: Expecting value: line 1 column 1 (char 0)
Error at index 10524: Expecting value: line 1 column 1 (char 0)
Error at index 10536: Expecting value: line 1 column 1 (char 0)
Error at index 10568: Expecting value: line 1 column 1 (char 0)
Error at index 10576: Expecting value: line 1 column 1 (char 0)
Error at index 10584: Expecting value: line 1 column 1 (char 0)
Error at index 10624: Expecting value: line 1 column 1 (char 0)
Error at index 10692: Expecting value: line 1 column 1 (char 0)
Error at index 10712: Expecting value: line 1 column 1 (char 0)
Error at index 10744: Expecting value: line 1 column 1 (char 0)
Error at index 10768: Expecting value: line 1 column 1 (char 0)
Error at index 10840: Expecting value: line 1 column 1 (char 0)
Error at index 10856: Expecting value: line 1 column 1 (char 0)
Error at index 10896: Expecting value: line 1 column 1 (char 0)
Error at index 10900: Expecting value: line 1 column 1 (char 0)
Error at index 10924: Expecting value: line 1 column 1 (char 0)
Error at index 10944: Expecting value: line 1 column 1 (char 0)
Error at index 11000: Expecting value: line 1 column 1 (char 0)
Error at index 11020: Expecting property name enclosed in double quotes: line 110 column 7 (char 2410)
Error at index 11036: Expecting value: line 1 column 1 (char 0)
Error at index 11048: Expecting value: line 1 column 1 (char 0)
Error at index 11072: Expecting value: line 1 column 1 (char 0)
Error at index 11080: Expecting value: line 1 column 1 (char 0)
Error at index 11088: Expecting value: line 1 column 1 (char 0)
Error at index 11092: Expecting value: line 1 column 1 (char 0)
Error at index 11104: Expecting value: line 1 column 1 (char 0)
Error at index 11136: Expecting value: line 1 column 1 (char 0)
Error at index 11140: Expecting value: line 1 column 1 (char 0)
Error at index 11200: Expecting value: line 1 column 1 (char 0)
Error at index 11212: Expecting value: line 1 column 1 (char 0)
Error at index 11244: Expecting value: line 1 column 1 (char 0)
Error at index 11252: Expecting value: line 1 column 1 (char 0)
Error at index 11256: Expecting value: line 1 column 1 (char 0)
Error at index 11280: Expecting value: line 1 column 1 (char 0)
Error at index 11292: Expecting value: line 1 column 1 (char 0)
Error at index 11308: Expecting value: line 1 column 1 (char 0)
Error at index 11336: Expecting value: line 1 column 1 (char 0)
Error at index 11340: Expecting value: line 1 column 1 (char 0)
Error at index 11344: Expecting value: line 1 column 1 (char 0)
Error at index 11352: Expecting value: line 1 column 1 (char 0)
Error at index 11356: Expecting value: line 1 column 1 (char 0)
Error at index 11392: Expecting value: line 1 column 1 (char 0)
Error at index 11412: Expecting value: line 1 column 1 (char 0)
Error at index 11416: Expecting value: line 1 column 1 (char 0)
Error at index 11432: Expecting value: line 1 column 1 (char 0)
Error at index 11440: Expecting value: line 1 column 1 (char 0)
Error at index 11476: Expecting value: line 1 column 1 (char 0)
Error at index 11484: Expecting value: line 1 column 1 (char 0)
Error at index 11504: Expecting ',' delimiter: line 122 column 1 (char 2280)
Error at index 11524: Expecting value: line 1 column 1 (char 0)
Error at index 11540: Expecting value: line 1 column 1 (char 0)
Error at index 11584: Expecting value: line 1 column 1 (char 0)
Error at index 11596: Expecting value: line 1 column 1 (char 0)
Error at index 11616: Expecting value: line 1 column 1 (char 0)
Error at index 11652: Expecting ',' delimiter: line 131 column 1 (char 2377)
Error at index 11672: Expecting value: line 1 column 1 (char 0)
Error at index 11696: Expecting value: line 1 column 1 (char 0)
Error at index 11704: Expecting ',' delimiter: line 130 column 2 (char 2250)
Error at index 11712: Expecting value: line 1 column 1 (char 0)
Error at index 11736: Expecting value: line 1 column 1 (char 0)
Error at index 11748: Expecting ',' delimiter: line 108 column 1 (char 1898)
Error at index 11760: Expecting value: line 1 column 1 (char 0)
Error at index 11780: Expecting value: line 1 column 1 (char 0)
Error at index 11788: Expecting value: line 1 column 1 (char 0)
Error at index 11804: Expecting value: line 1 column 1 (char 0)
Error at index 11812: Expecting value: line 1 column 1 (char 0)
Error at index 11824: Expecting value: line 1 column 1 (char 0)
Error at index 11836: Expecting value: line 1 column 1 (char 0)
Error at index 11840: Expecting value: line 1 column 1 (char 0)
Error at index 11864: Expecting value: line 1 column 1 (char 0)
Error at index 11880: Expecting value: line 1 column 1 (char 0)
Error at index 11884: Expecting value: line 1 column 1 (char 0)
Error at index 11924: Expecting value: line 1 column 1 (char 0)
Error at index 11932: Expecting value: line 1 column 1 (char 0)
Error at index 11984: Expecting value: line 1 column 1 (char 0)
Error at index 11988: Expecting value: line 1 column 1 (char 0)
Error at index 12000: Expecting value: line 1 column 1 (char 0)
Error at index 12012: Expecting value: line 1 column 1 (char 0)
Error at index 12020: Expecting value: line 1 column 1 (char 0)
Error at index 12088: Expecting value: line 1 column 1 (char 0)
Error at index 12104: Expecting value: line 1 column 1 (char 0)
Error at index 12108: Expecting value: line 1 column 1 (char 0)
Error at index 12128: Expecting value: line 1 column 1 (char 0)
Error at index 12140: Expecting value: line 1 column 1 (char 0)
Error at index 12148: Expecting value: line 1 column 1 (char 0)
Error at index 12156: Expecting value: line 1 column 1 (char 0)
Error at index 12160: Expecting value: line 1 column 1 (char 0)
Error at index 12168: Expecting value: line 1 column 1 (char 0)
Error at index 12180: Expecting value: line 1 column 1 (char 0)
Error at index 12204: Expecting value: line 1 column 1 (char 0)
Error at index 12240: Expecting value: line 1 column 1 (char 0)
Error at index 12256: Expecting value: line 1 column 1 (char 0)
Error at index 12272: Expecting value: line 1 column 1 (char 0)
Error at index 12288: Expecting value: line 1 column 1 (char 0)
Error at index 12364: Expecting value: line 1 column 1 (char 0)
Error at index 12476: Expecting value: line 1 column 1 (char 0)
Error at index 12480: Expecting value: line 1 column 1 (char 0)
Error at index 12484: Expecting value: line 1 column 1 (char 0)
Error at index 12488: Expecting value: line 1 column 1 (char 0)
Error at index 12532: Expecting value: line 1 column 1 (char 0)
Error at index 12544: Expecting value: line 1 column 1 (char 0)
Error at index 12580: Expecting value: line 1 column 1 (char 0)
Error at index 12592: Expecting value: line 1 column 1 (char 0)
Error at index 12596: Expecting value: line 1 column 1 (char 0)
Error at index 12608: Expecting value: line 1 column 1 (char 0)
Error at index 12656: Expecting value: line 1 column 1 (char 0)
Error at index 12668: Expecting value: line 1 column 1 (char 0)
Error at index 12684: Expecting value: line 1 column 1 (char 0)
Error at index 12724: Expecting value: line 1 column 1 (char 0)
Error at index 12728: Expecting value: line 1 column 1 (char 0)
Error at index 12736: Expecting value: line 1 column 1 (char 0)
Error at index 12740: Expecting value: line 1 column 1 (char 0)
Error at index 12768: Expecting value: line 1 column 1 (char 0)
Error at index 12776: Expecting value: line 1 column 1 (char 0)
Error at index 12808: Expecting value: line 1 column 1 (char 0)
Error at index 12820: Expecting value: line 1 column 1 (char 0)
Error at index 12840: Expecting value: line 1 column 1 (char 0)
Error at index 12844: Expecting value: line 1 column 1 (char 0)
Error at index 12848: Expecting value: line 1 column 1 (char 0)
Error at index 12856: Expecting value: line 1 column 1 (char 0)
Error at index 12888: Expecting value: line 1 column 1 (char 0)
Error at index 12908: Expecting value: line 1 column 1 (char 0)
Error at index 12932: Expecting value: line 1 column 1 (char 0)
Error at index 12948: Expecting value: line 1 column 1 (char 0)
Error at index 12980: Expecting value: line 1 column 1 (char 0)
Error at index 12988: Expecting value: line 1 column 1 (char 0)
Error at index 12992: Expecting value: line 1 column 1 (char 0)
Error at index 13000: Expecting ',' delimiter: line 90 column 1 (char 1717)
Error at index 13040: Expecting value: line 1 column 1 (char 0)
Error at index 13044: Expecting value: line 1 column 1 (char 0)
Error at index 13048: Expecting value: line 1 column 1 (char 0)
Error at index 13100: Expecting value: line 1 column 1 (char 0)
Error at index 13112: Expecting value: line 1 column 1 (char 0)
Error at index 13156: Expecting value: line 1 column 1 (char 0)
Error at index 13196: Expecting value: line 1 column 1 (char 0)
Error at index 13272: Expecting value: line 1 column 1 (char 0)
Error at index 13292: Expecting value: line 1 column 1 (char 0)
Error at index 13328: Expecting value: line 1 column 1 (char 0)
Error at index 13352: Expecting value: line 1 column 1 (char 0)
Error at index 13376: Expecting value: line 1 column 1 (char 0)
Error at index 13400: Expecting value: line 1 column 1 (char 0)
Error at index 13424: Expecting value: line 1 column 1 (char 0)
Error at index 13464: Expecting value: line 1 column 1 (char 0)
Error at index 13532: Expecting value: line 1 column 1 (char 0)
Error at index 13548: Expecting value: line 1 column 1 (char 0)
Error at index 13608: Expecting value: line 1 column 1 (char 0)
Error at index 13668: Expecting value: line 1 column 1 (char 0)
Error at index 13672: Expecting value: line 1 column 1 (char 0)
Error at index 13688: Expecting value: line 1 column 1 (char 0)
Error at index 13696: Expecting value: line 1 column 1 (char 0)
Error at index 13728: Expecting value: line 1 column 1 (char 0)
Error at index 13748: Expecting value: line 1 column 1 (char 0)
Error at index 13768: Expecting value: line 1 column 1 (char 0)
Error at index 13784: Expecting value: line 1 column 1 (char 0)
Error at index 13800: Expecting value: line 1 column 1 (char 0)
Error at index 13816: Expecting value: line 1 column 1 (char 0)
Error at index 13896: Expecting value: line 1 column 1 (char 0)
Error at index 13928: Expecting value: line 1 column 1 (char 0)
Error at index 13940: Expecting value: line 1 column 1 (char 0)
Error at index 14008: Expecting value: line 1 column 1 (char 0)
Error at index 14024: Expecting value: line 1 column 1 (char 0)
Error at index 14028: Expecting value: line 1 column 1 (char 0)
Error at index 14048: Expecting value: line 1 column 1 (char 0)
Error at index 14056: Expecting value: line 1 column 1 (char 0)
Error at index 14072: Expecting value: line 1 column 1 (char 0)
Error at index 14096: Expecting value: line 1 column 1 (char 0)
Error at index 14120: Expecting value: line 1 column 1 (char 0)
Error at index 14188: Expecting value: line 1 column 1 (char 0)
Error at index 14240: Expecting value: line 1 column 1 (char 0)
Error at index 14252: Expecting value: line 1 column 1 (char 0)
Error at index 14300: Expecting value: line 1 column 1 (char 0)
Error at index 14328: Expecting value: line 1 column 1 (char 0)
Error at index 14344: Expecting value: line 1 column 1 (char 0)
Error at index 14376: Expecting value: line 1 column 1 (char 0)
Error at index 14452: Expecting value: line 1 column 1 (char 0)
Error at index 14460: Expecting value: line 1 column 1 (char 0)
Error at index 14496: Expecting value: line 1 column 1 (char 0)
Error at index 14516: Expecting value: line 1 column 1 (char 0)
Error at index 14580: Expecting value: line 1 column 1 (char 0)
Error at index 14656: Expecting value: line 1 column 1 (char 0)
Error at index 14676: Expecting value: line 1 column 1 (char 0)
Error at index 14796: Expecting value: line 1 column 1 (char 0)
Error at index 14940: Expecting value: line 1 column 1 (char 0)
Error at index 15004: Expecting value: line 1 column 1 (char 0)
Error at index 15012: Expecting value: line 1 column 1 (char 0)
Error at index 15036: Expecting value: line 1 column 1 (char 0)
Error at index 15048: Expecting value: line 1 column 1 (char 0)
Error at index 15056: Expecting value: line 1 column 1 (char 0)
Error at index 15068: Expecting value: line 1 column 1 (char 0)
Error at index 15104: Expecting value: line 1 column 1 (char 0)
Error at index 15108: Expecting value: line 1 column 1 (char 0)
Error at index 15120: Expecting value: line 1 column 1 (char 0)
Error at index 15140: Expecting value: line 1 column 1 (char 0)
Error at index 15184: Expecting value: line 1 column 1 (char 0)
Error at index 15188: Expecting value: line 1 column 1 (char 0)
Error at index 15200: Expecting value: line 1 column 1 (char 0)
Error at index 15248: Expecting value: line 1 column 1 (char 0)
Error at index 15360: Expecting value: line 1 column 1 (char 0)
Error at index 15364: Expecting value: line 1 column 1 (char 0)
Error at index 15448: Expecting value: line 1 column 1 (char 0)
Error at index 15464: Expecting value: line 1 column 1 (char 0)
Error at index 15480: Expecting value: line 1 column 1 (char 0)
Error at index 15500: Expecting value: line 1 column 1 (char 0)
Error at index 15520: Expecting value: line 1 column 1 (char 0)
Error at index 15576: Expecting value: line 1 column 1 (char 0)
Error at index 15616: Expecting value: line 1 column 1 (char 0)
Error at index 15632: Expecting value: line 1 column 1 (char 0)
Error at index 15644: Expecting value: line 1 column 1 (char 0)
Error at index 15664: Expecting value: line 1 column 1 (char 0)
Error at index 15676: Expecting value: line 1 column 1 (char 0)
Error at index 15684: Expecting value: line 1 column 1 (char 0)
Error at index 15712: Expecting value: line 1 column 1 (char 0)
Error at index 15728: Expecting value: line 1 column 1 (char 0)
Error at index 15748: Expecting value: line 1 column 1 (char 0)
Error at index 15800: Expecting value: line 1 column 1 (char 0)
Error at index 15820: Expecting value: line 1 column 1 (char 0)
Error at index 15868: Expecting value: line 1 column 1 (char 0)
Error at index 15980: Expecting value: line 1 column 1 (char 0)
Error at index 15992: Expecting value: line 1 column 1 (char 0)
Error at index 16008: Expecting value: line 1 column 1 (char 0)
Error at index 16032: Expecting value: line 1 column 1 (char 0)
Error at index 16036: Expecting value: line 1 column 1 (char 0)
Error at index 16060: Expecting value: line 1 column 1 (char 0)
Error at index 16108: Expecting value: line 1 column 1 (char 0)
Error at index 16132: Expecting value: line 1 column 1 (char 0)
Error at index 16144: Expecting value: line 1 column 1 (char 0)
Error at index 16160: Expecting value: line 1 column 1 (char 0)
Error at index 16164: Expecting value: line 1 column 1 (char 0)
Error at index 16204: Expecting value: line 1 column 1 (char 0)
Error at index 16216: Expecting value: line 1 column 1 (char 0)
Error at index 16260: Expecting value: line 1 column 1 (char 0)
Error at index 16272: Expecting value: line 1 column 1 (char 0)
Error at index 16280: Expecting value: line 1 column 1 (char 0)
Error at index 16292: Expecting value: line 1 column 1 (char 0)
Error at index 16308: Expecting ',' delimiter: line 131 column 2 (char 2184)
Error at index 16312: Expecting value: line 1 column 1 (char 0)
Error at index 16368: Expecting ',' delimiter: line 133 column 2 (char 2221)
Error at index 16408: Expecting value: line 1 column 1 (char 0)
Error at index 16432: Expecting value: line 1 column 1 (char 0)
Error at index 16452: Expecting value: line 1 column 1 (char 0)
Error at index 16480: Expecting value: line 1 column 1 (char 0)
Error at index 16556: Expecting value: line 1 column 1 (char 0)
Error at index 16600: Expecting value: line 1 column 1 (char 0)
Error at index 16624: Expecting value: line 1 column 1 (char 0)
Error at index 16628: Expecting value: line 1 column 1 (char 0)
Error at index 16668: Expecting value: line 1 column 1 (char 0)
Error at index 16688: Expecting value: line 1 column 1 (char 0)
Error at index 16736: Expecting value: line 1 column 1 (char 0)
Error at index 16744: Expecting value: line 1 column 1 (char 0)
Error at index 16828: Expecting value: line 1 column 1 (char 0)
Error at index 16884: Expecting value: line 1 column 1 (char 0)
Error at index 16948: Expecting value: line 1 column 1 (char 0)
Error at index 16960: Expecting value: line 1 column 1 (char 0)
Error at index 16964: Expecting value: line 1 column 1 (char 0)
Error at index 16968: Expecting value: line 1 column 1 (char 0)
Error at index 16984: Expecting value: line 1 column 1 (char 0)
Error at index 16988: Expecting value: line 1 column 1 (char 0)
Error at index 17020: Expecting value: line 1 column 1 (char 0)
Error at index 17100: Expecting value: line 1 column 1 (char 0)
Error at index 17220: Expecting value: line 1 column 1 (char 0)
Error at index 17232: Expecting value: line 1 column 1 (char 0)
Error at index 17284: Expecting value: line 1 column 1 (char 0)
Error at index 17288: Expecting ',' delimiter: line 115 column 1 (char 1723)
Error at index 17324: Expecting value: line 1 column 1 (char 0)
Error at index 17332: Expecting value: line 1 column 1 (char 0)
Error at index 17360: Expecting value: line 1 column 1 (char 0)
Error at index 17364: Expecting value: line 1 column 1 (char 0)
Error at index 17376: Expecting value: line 1 column 1 (char 0)
Error at index 17428: Expecting value: line 1 column 1 (char 0)
Error at index 17524: Expecting value: line 1 column 1 (char 0)
Error at index 17536: Expecting value: line 1 column 1 (char 0)
Error at index 17568: Expecting value: line 1 column 1 (char 0)
Error at index 17700: Expecting value: line 1 column 1 (char 0)
Error at index 17716: Expecting value: line 1 column 1 (char 0)
Error at index 17744: Expecting ',' delimiter: line 69 column 1 (char 996)
Error at index 17764: Expecting value: line 1 column 1 (char 0)
Error at index 17836: Expecting value: line 1 column 1 (char 0)
Error at index 17860: Expecting value: line 1 column 1 (char 0)
Error at index 17868: Expecting value: line 1 column 1 (char 0)
Error at index 17916: Expecting value: line 1 column 1 (char 0)
Error at index 17956: Expecting value: line 1 column 1 (char 0)
Error at index 17976: Expecting value: line 1 column 1 (char 0)
Error at index 18024: Expecting value: line 1 column 1 (char 0)
Error at index 18120: Expecting value: line 1 column 1 (char 0)
Error at index 18124: Expecting value: line 1 column 1 (char 0)
Error at index 18128: Expecting value: line 1 column 1 (char 0)
Error at index 18168: Expecting value: line 1 column 1 (char 0)
Error at index 18204: Expecting value: line 1 column 1 (char 0)
Error at index 18208: Expecting value: line 1 column 1 (char 0)
Error at index 18216: Expecting value: line 1 column 1 (char 0)
Error at index 18244: Expecting value: line 1 column 1 (char 0)
Error at index 18260: Expecting value: line 1 column 1 (char 0)
Error at index 18268: Expecting value: line 1 column 1 (char 0)
Error at index 18292: Expecting value: line 1 column 1 (char 0)
Error at index 18312: Expecting value: line 1 column 1 (char 0)
Error at index 18328: Expecting value: line 1 column 1 (char 0)
Error at index 18400: Expecting value: line 1 column 1 (char 0)
Error at index 18420: Expecting value: line 1 column 1 (char 0)
Error at index 18520: Expecting value: line 1 column 1 (char 0)
Error at index 18532: Expecting value: line 1 column 1 (char 0)
Error at index 18580: Expecting value: line 1 column 1 (char 0)
Error at index 18648: Expecting value: line 1 column 1 (char 0)
Error at index 18692: Expecting value: line 1 column 1 (char 0)
Error at index 18696: Expecting value: line 1 column 1 (char 0)
Error at index 18740: Expecting ',' delimiter: line 76 column 1 (char 1083)
Error at index 18760: Expecting value: line 1 column 1 (char 0)
Error at index 18776: Expecting value: line 1 column 1 (char 0)
Error at index 18780: Expecting value: line 1 column 1 (char 0)
Error at index 18788: Expecting value: line 1 column 1 (char 0)
Error at index 18792: Expecting ',' delimiter: line 89 column 2 (char 1418)
Error at index 18808: Expecting value: line 1 column 1 (char 0)
Error at index 18848: Expecting ',' delimiter: line 76 column 1 (char 1179)
Error at index 19088: Expecting value: line 1 column 1 (char 0)
Error at index 19092: Expecting value: line 1 column 1 (char 0)
Error at index 19108: Expecting value: line 1 column 1 (char 0)
Error at index 19184: Expecting value: line 1 column 1 (char 0)
Error at index 19248: Expecting value: line 1 column 1 (char 0)
Error at index 19256: Expecting value: line 1 column 1 (char 0)
Error at index 19300: Expecting value: line 1 column 1 (char 0)
Error at index 19308: Expecting value: line 1 column 1 (char 0)
Error at index 19316: Expecting value: line 1 column 1 (char 0)
Error at index 19360: Expecting ',' delimiter: line 96 column 2 (char 1774)
"""

In [62]:
import ollama
import pandas as pd
import json
import time
import re
from tqdm.notebook import tqdm

# =====================
# CONFIG
# =====================
INPUT_FILE = "rundi_extracted_cleaned_full.txt"
ORIGINAL_CSV = f"rundi_llm_tagged_gemma4_fromP1-C1.csv"
OUTPUT_FIXED_CSV = f"rundi_llm_tagged_gemma4_fromP1-C1_FIXED.csv"

RAW_LOG_FILE = OUTPUT_FIXED_CSV.replace(".csv", "_RAW_LOG.csv")

START_COLUMN = 'P70-C1' ## <<<------------------------------- pick-up where we left off

SYSTEM_PROMPT = """
Extract Rundi dictionary data into a JSON array labeled "entries". 

For each <entry>, extract:
- WORD: Headword only.
- AFFIX: Prefix/suffix with a SINGLE hyphen (e.g., "-ye", "imy-"). 
- NOUN_CLASS: Numeric classes (e.g., "9|10", "11"). Skip if not found.
- DESCRIPTION: A concise French definition. Omit examples and long ethnographic notes. 
- POS: Predict from the DESCRIPTION (e.g., "v.", "n.", "adj.")

Rules:
1. Do not skip the POS tag!
2. DESCRIPTION must be a short gloss, not the entire entry.
3. Maintain the order of entries provided.
4. No explanations. No text outside JSON.
"""

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    raw_chunks = [e.strip() for e in f.read().split('\n\n') if len(e.strip()) > 3]

processed_entries = []
current_column = "UNKNOWN"

# If START_COLUMN is set, start_processing is False until we find the marker
start_processing = True if not START_COLUMN else False

for chunk in raw_chunks:
    if chunk.upper().startswith("COLUMN-START"):
        current_column = chunk
        # Check if this marker matches our starting point
        if START_COLUMN and START_COLUMN in chunk:
            start_processing = True
            print(f"Found start marker: {chunk}. Beginning extraction...")
    else:
        # Only append if we have passed the start marker
        if start_processing:
            processed_entries.append((chunk, current_column))

TOTAL_ENTRIES = len(processed_entries)

# =====================
# FAILED BATCHES
# =====================
failed_entry_indices = sorted(set(map(int, re.findall(r'index (\d+)', ERROR_LOG))))
failed_batches = sorted(set(i // BATCH_SIZE for i in failed_entry_indices))

print("Failed batches:", failed_batches)

# =====================
# REPAIR LOOP (FIXED)
# =====================

open(OUTPUT_FIXED_CSV, 'w', encoding='utf-8').close()
header_flag = 1
for batch_idx in tqdm(failed_batches, desc="Repairing Failed Batches", unit="batch"):

    i = batch_idx * BATCH_SIZE

    # 🔥 FIX #1: THIS WAS MISSING (CRITICAL)
    batch_tuples = processed_entries[i:i + BATCH_SIZE]

    if not batch_tuples:
        print(f"⚠️ Empty batch slice at {batch_idx}")
        continue

    batch_str = "\n".join([
        f"<entry id='{idx}'>{text}</entry>"
        for idx, (text, col) in enumerate(batch_tuples)
    ])

    items = []
    raw_output = ""

    for attempt in range(3):
        try:
            response = ollama.chat(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {
                        'role': 'user',
                        'content': f"""
Return ONLY valid JSON.

{batch_str}
"""
                    }
                ],
                # 🔥 FIX #2: REMOVE JSON MODE (CRITICAL FOR GEMMA)
                options={
                    'temperature': 0,
                    'num_ctx': 4096,
                    'num_predict': 2048,
                    'top_k': 1,
                    'top_p': 0.01
                }
            )

            content = response['message'].get('content', '')
            raw_output = content

            if not content.strip():
                print(f"⚠️ Empty output batch {batch_idx}, retry {attempt+1}")
                time.sleep(1)
                continue

            # safe parse
            try:
                batch_data = json.loads(content)
            except:
                match = re.search(r'\{.*\}', content, re.DOTALL)
                batch_data = json.loads(match.group(0)) if match else None

            if not batch_data:
                print(f"⚠️ Bad JSON batch {batch_idx}")
                continue

            extracted_list = batch_data.get('entries', [])
            if not isinstance(extracted_list, list):
                extracted_list = [extracted_list]

            batch_results = []

            for idx, (original_text, source_col) in enumerate(batch_tuples):

                item = extracted_list[idx] if idx < len(extracted_list) else None

                if item:
                    item['PAGE-COLUMN'] = "-".join(source_col.split("-")[2:4])
                    item['FULL_ENTRY'] = original_text
                    item.pop('id', None)
                    item["GLOBAL_INDEX"] = i + idx
                    batch_results.append(item)

            if batch_results:
                items = batch_results
                break

        except Exception as e:
            if attempt == 2:
                print(f"❌ Batch {batch_idx} failed: {e}")
            else:
                time.sleep(1)

    # fallback (never lose data)
    if not items:
        print(f"⚠️ Fallback batch {batch_idx}")
        for idx, (text, col) in enumerate(batch_tuples):
            items.append({
                "WORD": None,
                "AFFIX": None,
                "NOUN_CLASS": None,
                "DESCRIPTION": None,
                "POS": None,
                "PAGE-COLUMN": "-".join(col.split("-")[2:4]),
                "FULL_ENTRY": text,
                "GLOBAL_INDEX": i + idx
            })

    # print('batch:',batch_idx,items,'\n')
    
    df = pd.DataFrame(items)
    # append results to CSV
    df.to_csv(
        OUTPUT_FIXED_CSV,
        mode='a',
        index=False,
        header=(header_flag == 1),
        encoding='utf-8'
    )
    # =====================
    # STORE RAW LOG (APPEND MODE)
    # =====================
    
    pd.DataFrame([{
        "GLOBAL_INDEX": i,
        "RAW_OUTPUT": raw_output
    }]).to_csv(
        RAW_LOG_FILE,
        mode='a',
        index=False,
        header=(header_flag == 1),
        encoding='utf-8'
    )
    header_flag = 0

print("\n✅ FIX COMPLETE")            

# start column for failed batch run: P70-C1

Found start marker: COLUMN-START-P70-C1-DID. Beginning extraction...
Failed batches: [632, 636, 648, 653, 673, 680, 682, 700, 709, 714, 720, 745, 784, 839, 846, 877, 900, 907, 923, 934, 936, 940, 957, 958, 976, 986, 988, 989, 999, 1004, 1007, 1017, 1019, 1033, 1044, 1051, 1075, 1078, 1082, 1087, 1093, 1107, 1122, 1126, 1144, 1148, 1190, 1213, 1219, 1236, 1290, 1294, 1302, 1336, 1355, 1366, 1369, 1373, 1378, 1380, 1394, 1402, 1403, 1407, 1416, 1431, 1436, 1471, 1515, 1528, 1532, 1544, 1564, 1568, 1580, 1586, 1600, 1608, 1628, 1633, 1635, 1641, 1643, 1647, 1658, 1682, 1683, 1687, 1714, 1720, 1721, 1726, 1731, 1746, 1748, 1755, 1762, 1771, 1775, 1794, 1798, 1811, 1815, 1818, 1864, 1871, 1874, 1877, 1895, 1904, 1917, 1919, 1938, 1953, 1970, 1975, 1977, 1998, 1999, 2006, 2026, 2041, 2045, 2046, 2048, 2053, 2058, 2074, 2079, 2080, 2092, 2100, 2102, 2103, 2109, 2134, 2146, 2150, 2152, 2159, 2161, 2166, 2177, 2189, 2194, 2195, 2196, 2199, 2203, 2209, 2213, 2226, 2234, 2240, 2248, 2257, 2262, 2

Repairing Failed Batches:   0%|          | 0/516 [00:00<?, ?batch/s]

❌ Batch 1402 failed: Expecting ',' delimiter: line 84 column 6 (char 1776)
⚠️ Fallback batch 1402
❌ Batch 1407 failed: Expecting property name enclosed in double quotes: line 69 column 3 (char 1466)
⚠️ Fallback batch 1407
⚠️ Empty output batch 2100, retry 1
❌ Batch 2100 failed: Expecting ',' delimiter: line 65 column 6 (char 1677)
⚠️ Fallback batch 2100
⚠️ Empty output batch 2189, retry 1
⚠️ Empty output batch 2189, retry 2
⚠️ Empty output batch 2189, retry 3
⚠️ Fallback batch 2189
❌ Batch 3045 failed: Expecting ',' delimiter: line 65 column 6 (char 1413)
⚠️ Fallback batch 3045
⚠️ Empty output batch 3437, retry 1
⚠️ Empty output batch 3437, retry 2
⚠️ Empty output batch 3437, retry 3
⚠️ Fallback batch 3437
❌ Batch 3560 failed: Expecting ',' delimiter: line 37 column 6 (char 863)
⚠️ Fallback batch 3560
⚠️ Empty output batch 3812, retry 1
⚠️ Empty output batch 3812, retry 2
⚠️ Empty output batch 3812, retry 3
⚠️ Fallback batch 3812
⚠️ Bad JSON batch 3928
⚠️ Bad JSON batch 3928
⚠️ Bad JSO

#### Check diff

In [134]:
import pandas as pd
from csv_diff import load_csv, compare

ORIGINAL_CSV = 'rundi_llm_tagged_WITH_INDEX.csv'
UPDATED_CSV = 'rundi_llm_tagged_FINAL.csv'

# 1. Run the diff
# Make sure to specify the 'key' (unique identifier) for your rows 
# e.g., 'WORD' or 'FULL_ENTRY'
diff = compare(
    load_csv(open(ORIGINAL_CSV, encoding='utf-8'), key="GLOBAL_INDEX"),
    load_csv(open(UPDATED_CSV, encoding='utf-8'), key="GLOBAL_INDEX")
)

def diff_to_dataframe(diff_dict):
    rows = []
    
    # Process Added records
    for item in diff_dict.get('added', []):
        row = item.copy()
        row['diff_status'] = 'ADDED'
        rows.append(row)
        
    # Process Removed records
    for item in diff_dict.get('removed', []):
        row = item.copy()
        row['diff_status'] = 'REMOVED'
        rows.append(row)
        
    # Process Changed records
    for item in diff_dict.get('changed', []):
        # item['key'] is the unique ID, item['changes'] is a dict of {field: [old, new]}
        row = {'WORD': item['key'], 'diff_status': 'CHANGED'}
        for field, change_values in item['changes'].items():
            row[f'{field}_OLD'] = change_values[0]
            row[f'{field}_NEW'] = change_values[1]
        rows.append(row)
        
    return pd.DataFrame(rows)

# 2. Generate the DataFrame
df_diff = diff_to_dataframe(diff)

# Optional: Export to review what changed
df_diff.to_csv("diff.csv", index=False)

In [135]:
df_diff[df_diff['diff_status']=="CHANGED"]

,WORD,diff_status,WORD_OLD,WORD_NEW,DESCRIPTION_OLD,DESCRIPTION_NEW,POS_OLD,POS_NEW,AFFIX_OLD,AFFIX_NEW,NOUN_CLASS_OLD,NOUN_CLASS_NEW
0,99,CHANGED,,ukurâho,,-râho,,v.,NaN,NaN,NaN,NaN
1,112,CHANGED,,nyawïiyahura,,sorte de papillon,,n.,NaN,NaN,NaN,NaN
2,113,CHANGED,,kwâhuranya,,couper,,v.,,-nije,NaN,NaN
3,114,CHANGED,,kwîyahurira,,"se suicider à cause de, pour",,v.,,-iye,NaN,NaN
4,115,CHANGED,,umwâhuro,,une rigole,,n.,,imy-,,3|4
...,...,...,...,...,...,...,...,...,...,...,...,...
2660,19363,CHANGED,,vertige,,prendre des manières européens,NaN,NaN,NaN,NaN,NaN,NaN
2661,19408,CHANGED,,kuzyêgenya,,produire un bruit,,v.,,-nije,NaN,NaN
2662,19409,CHANGED,,ikizyěte,,vessie des bovidés,,n.,,ibi,NaN,NaN
2663,19410,CHANGED,,kuzyôgra,,"gazouiller, jacasser",,v.,,-ye,NaN,NaN


In [1]:
!export OLLAMA_NUM_PARALLEL=4

In [1]:
import ollama
import pandas as pd
import json
import time
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


MODEL = "gemma4:e2b"
INPUT_FILE = "rundi_extracted_cleaned_full.txt"
OUTPUT_CSV = f"rundi_llm_tagged_{MODEL.split(':')[0]}_fast.csv"

BATCH_SIZE = 4
MAX_WORKERS = 1          # parallelism (set 1 to disable)
WRITE_BUFFER = 1        # write every N batches

SYSTEM_PROMPT = """
Return a JSON object with a single key "entries" containing an array of objects.
For every <entry> provided, extract:
- WORD: The Rundi headword.
- PLURAL: The prefix or suffix containing a hyphen.
- NOUN_CLASS: The numeric class pair (e.g., "3|4").
- DESCRIPTION: The first French definition.
- POS: The part-of-speech abbreviation.

If a field is missing, use null. Do not skip any entries.
"""


# =====================
# SAFE JSON LOAD
# =====================
def safe_json_load(text):
    try:
        return json.loads(text)
    except:
        return None


# =====================
# PROCESS ONE BATCH
# =====================
def process_batch(batch, i):
    batch_str = "\n".join([
        f"<entry id='{idx}'>{text}</entry>"
        for idx, text in enumerate(batch)
    ])

    for attempt in range(2):
        try:
            response = ollama.chat(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': f"Extract entries:\n{batch_str}"}
                ],
                format='json',
                options={
                    'temperature': 0,
                    'num_ctx': 2048,     # 🔥 reduced
                    'num_predict': 256,  # 🔥 reduced (big speedup)
                    'top_k': 1,
                    'top_p': 0.0
                }
            )

            content = response['message']['content']
            data = safe_json_load(content)

            if data is None:
                continue

            # same logic as your original
            if isinstance(data, dict) and 'entries' in data:
                return data['entries']
            elif isinstance(data, list):
                return data
            else:
                return [data]

        except Exception as e:
            if attempt == 1:
                print(f"⚠️ Error batch {i}: {e}")
            else:
                time.sleep(1)

    return []


# =====================
# MAIN FUNCTION
# =====================
def fast_batch_process():
    # 1. Load
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        all_entries = [
            e.strip() for e in f.read().split('\n\n')
            if len(e.strip()) > 3 and not e.strip().upper().startswith("COLUMN-START")
        ]

    # clear output
    open(OUTPUT_CSV, 'w').close()

    buffer = []
    header_written = False

    # 2. Create batches (same logic)
    batches = [
        all_entries[i:i+BATCH_SIZE]
        for i in range(0, len(all_entries), BATCH_SIZE)
    ]

    # warmup (faster first real call)
    ollama.chat(model=MODEL, messages=[{'role': 'user', 'content': 'hi'}])

    # 3. Parallel or sequential
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process_batch, batch, i): i
            for i, batch in enumerate(batches)
        }

        pbar = tqdm(total=len(futures), desc="Processing Batches")

        for future in as_completed(futures):
            items = future.result()

            if items:
                buffer.extend(items)

            # 🔥 buffered writing (much faster)
            if len(buffer) >= WRITE_BUFFER:
                pd.DataFrame(buffer).to_csv(
                    OUTPUT_CSV,
                    mode='a',
                    index=False,
                    header=not header_written,
                    encoding='utf-8'
                )
                header_written = True
                buffer.clear()

            pbar.update(1)

    # final flush
    if buffer:
        pd.DataFrame(buffer).to_csv(
            OUTPUT_CSV,
            mode='a',
            index=False,
            header=not header_written,
            encoding='utf-8'
        )

    print("\n✅ Extraction complete.")


# =====================
# RUN
# =====================
fast_batch_process()

Processing Batches:   0%|          | 0/4854 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Osaurus.ai

In [18]:
from openai import OpenAI
import pandas as pd
import json
import time
import os
from tqdm.notebook import tqdm

# --- CONFIG ---
MODEL = "gemma-4-e2b-it-4bit"
INPUT_FILE = "rundi_extracted_cleaned_full.txt"
OUTPUT_CSV = "rundi_llm_tagged_gemma4_osaurus.csv"
BATCH_SIZE = 4

EXPECTED_KEYS = ["WORD", "PLURAL", "NOUN_CLASS", "DESCRIPTION", "POS"]

client = OpenAI(base_url="http://127.0.0.1:1337/v1", api_key="osaurus")

SYSTEM_PROMPT = """
Return a JSON object with a single key "entries" containing an array of objects.

For EVERY <entry>:
- WORD: The Rundi headword
- PLURAL: The prefix or suffix
- NOUN_CLASS: The numeric class pair (e.g., "3|4")
- DESCRIPTION: The first French definition sentence
- POS: The part-of-speech abbreviation

RULES:
- Do not skip entries
- If missing → null
- Output MUST be valid JSON
- No markdown
- No text outside JSON
"""

# --- CLEAN START ---
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)


# --- LOAD DATA ---
def load_entries():
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        return [
            e.strip()
            for e in f.read().split("\n\n")
            if len(e.strip()) > 10
            and not e.strip().upper().startswith("COLUMN-START")
        ]


# --- NORMALIZE ---
def normalize(entry):
    return {
        "WORD": entry.get("WORD"),
        "PLURAL": entry.get("PLURAL"),
        "NOUN_CLASS": entry.get("NOUN_CLASS"),
        "DESCRIPTION": entry.get("DESCRIPTION"),
        "POS": entry.get("POS"),
    }


# --- MAIN PIPELINE ---
def extract_tags_osaurus():
    all_entries = load_entries()

    final_results = []
    batch_count = 0

    for i in tqdm(range(0, len(all_entries), BATCH_SIZE), desc="Processing", unit="batch"):
        batch = all_entries[i : i + BATCH_SIZE]
        batch_count += 1

        # 🔥 EXACTLY like Ollama: ID-tagged XML
        batch_str = "\n".join([
            f"<entry id='{idx}'>{text}</entry>"
            for idx, text in enumerate(batch)
        ])

        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {
                            "role": "user",
                            "content": f"Extract EVERY entry into JSON:\n{batch_str}"
                        }
                    ],
                    temperature=0,
                    top_p=0.01,   # 🔥 mimic ollama
                )

                content = response.choices[0].message.content.strip()

                # --- STRICT JSON PARSE ---
                batch_data = json.loads(content)

                # --- SAME LOGIC AS OLLAMA ---
                if isinstance(batch_data, dict) and "entries" in batch_data:
                    items = batch_data["entries"]
                elif isinstance(batch_data, list):
                    items = batch_data
                else:
                    items = [batch_data]

                # Normalize + enforce schema
                for item in items:
                    final_results.append(normalize(item))

                break  # success

            except Exception as e:
                if attempt == 2:
                    print(f"Error: {e}\n❌ Skipping batch {i} after 3 failures")
                else:
                    time.sleep(2)

        # --- SAVE EVERY EVEN BATCH ---
        if batch_count % 2 == 0:
            pd.DataFrame(final_results, columns=EXPECTED_KEYS).to_csv(
                OUTPUT_CSV, index=False
            )

    # --- FINAL SAVE ---
    df = pd.DataFrame(final_results, columns=EXPECTED_KEYS)
    df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n✅ Done. Total entries: {len(df)}")

In [19]:
start_time = time.perf_counter()

extract_tags_osaurus()

end_time = time.perf_counter()

print(f'Time taken to OCR: {(end_time-start_time)/60:.2f} min')

Processing:   0%|          | 0/4853 [00:00<?, ?batch/s]

Error: Expecting value: line 1 column 1 (char 0)
❌ Skipping batch 0 after 3 failures


KeyboardInterrupt: 

# Manual Tagging

In [58]:
import re
import pandas as pd

def extract_with_ocr_fixes(text_block):
    # Messy NC: digits mixed with garbage, or single digits
    nc_pattern = r"\d+[^, ]*?\d*[a-z]?"
    # POS tags to exclude from being identified as Plural/NC
    pos_tags = ["v.", "n.", "adj.", "v", "n", "V", "N", "adv.", "interj.", "inv.", "(inv.)"]
    
    extracted_data = []
    entries = [e.strip() for e in text_block.split('\n\n') if len(e.strip()) > 10]

    for entry in entries:
        if "COLUMN-START" in entry: continue
        
        full_entry = " ".join(entry.split())
        
        # 1. Extract WORD (everything before first comma)
        word_match = re.match(r"^([^,]+),", full_entry)
        if not word_match: continue
        word = word_match.group(1).strip()
        
        # The 'Working Area' is everything after the first comma
        remainder = full_entry[len(word)+1:].strip()
        
        plural = None
        noun_class = None
        
        # 2. Extract PLURAL & NC (Looking only at the start of the remainder)
        # We split the first few chunks to find metadata
        meta_chunks = [c.strip() for c in remainder.split(',', 3)]
        
        current_idx = 0
        while current_idx < len(meta_chunks) and current_idx < 2:
            chunk = meta_chunks[current_idx]
            
            # Check for Noun Class (contains a digit)
            if re.search(r"\d", chunk) and not chunk.startswith('('):
                nc_match = re.search(nc_pattern, chunk)
                if nc_match:
                    noun_class = nc_match.group(0)
                    # If we found NC in the second slot, the first slot was likely the plural
                    if current_idx == 1 and not plural:
                        plural = meta_chunks[0]
                    current_idx += 1
                    continue
            
            # Check for Plural (has hyphen or is known prefix)
            if '-' in chunk or any(chunk.startswith(pre) for pre in ['imy', 'utw', 'bany']):
                if chunk.lower() not in pos_tags:
                    plural = chunk
            
            current_idx += 1

        # 3. Extract DESCRIPTION
        # Rule: Find the first segment that doesn't start with '(', isn't a digit, 
        # and isn't the plural/nc we already found.
        description = None
        desc_parts = [p.strip() for p in remainder.split(',')]
        
        for p in desc_parts:
            # Clean part of POS tags for checking
            clean_p = p.lower().strip('.')
            
            if not p or p.startswith('(') or p[0].isdigit() or p == plural or p == noun_class or clean_p in pos_tags:
                continue
            
            # Once we find the start, take everything until the first full stop
            # Rejoin the rest of the entry to ensure we don't lose text after a comma
            start_pos = remainder.find(p)
            full_tail = remainder[start_pos:]
            sentence_match = re.search(r"(.*?[.!?])(?:\s|$|\|\|)", full_tail)
            
            description = sentence_match.group(1).strip() if sentence_match else p
            break

        # 4. Extract POS
        pos_match = re.search(r"\b(" + "|".join([re.escape(t) for t in pos_tags]) + r")\b\.?$", full_entry)
        pos = pos_match.group(1) if pos_match else None

        extracted_data.append({
            "WORD": word,
            "PLURAL": plural,
            "NOUN_CLASS": noun_class,
            "DESCRIPTION": description,
            "POS": pos,
            "FULL_ENTRY": entry
        })
            
    return pd.DataFrame(extracted_data)

In [60]:

with open("rundi_extracted_cleaned_full.txt", "r") as f:
    text = f.read()
df = extract_improved_rundi(text)
df.to_csv("rundi_tagged_regex.csv", index=False)

# Final Cleanup

## Fix failed batches

In [16]:
# some batches failed, we need to keep track of them better, so i'm adding a global index column to the result CSV

INPUT_FILE = "rundi_extracted_cleaned_full.txt"
LLM_CSV = "rundi_llm_tagged_gemma4_fromP1-C1.csv"
OUTPUT_CSV = "rundi_llm_tagged_WITH_INDEX.csv"

START_COLUMN = "P1-C1"

# =====================
# STEP 1: REBUILD processed_entries
# =====================
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    raw_chunks = [e.strip() for e in f.read().split('\n\n') if len(e.strip()) > 3]

processed_entries = []
current_column = "UNKNOWN"
start_processing = True if not START_COLUMN else False

for chunk in raw_chunks:
    if chunk.upper().startswith("COLUMN-START"):
        current_column = chunk
        if START_COLUMN and START_COLUMN in chunk:
            start_processing = True
    else:
        if start_processing:
            processed_entries.append((chunk, current_column))

TOTAL = len(processed_entries)
print(f"Total entries: {TOTAL}")

# =====================
# STEP 2: BUILD LOOKUP
# =====================
entry_to_index = {}
for idx, (text, col) in enumerate(processed_entries):
    entry_to_index[text] = idx

# =====================
# STEP 3: LOAD LLM CSV
# =====================
df = pd.read_csv(LLM_CSV, usecols=['WORD','AFFIX','NOUN_CLASS','DESCRIPTION','POS','PAGE-COLUMN','FULL_ENTRY'])

# ensure FULL_ENTRY exists
if "FULL_ENTRY" not in df.columns:
    raise ValueError("FULL_ENTRY column missing in LLM CSV")

# =====================
# STEP 4: ASSIGN GLOBAL INDEX
# =====================
df["GLOBAL_INDEX"] = df["FULL_ENTRY"].map(entry_to_index)

# =====================
# STEP 5: BUILD FULL DATASET (NO DROPS)
# =====================
full_rows = []

# create reverse lookup for fast fill
existing_map = {
    row["GLOBAL_INDEX"]: row
    for _, row in df.dropna(subset=["GLOBAL_INDEX"]).iterrows()
}

for idx, (text, col) in enumerate(processed_entries):
    if idx in existing_map:
        row = existing_map[idx].to_dict()
    else:
        # missing entry → create empty row
        row = {
            "WORD": None,
            "AFFIX": None,
            "NOUN_CLASS": None,
            "DESCRIPTION": None,
            "POS": None,
            "FULL_ENTRY": text,
        }

    # always enforce correct values
    row["FULL_ENTRY"] = text
    row["PAGE-COLUMN"] = "-".join(col.split("-")[2:4])
    row["GLOBAL_INDEX"] = idx

    full_rows.append(row)

# =====================
# STEP 6: FINAL DF
# =====================
final_df = pd.DataFrame(full_rows)
final_df = final_df.sort_values("GLOBAL_INDEX")

# reorder columns
cols = [
    "GLOBAL_INDEX",
    "WORD",
    "AFFIX",
    "NOUN_CLASS",
    "DESCRIPTION",
    "POS",
    "PAGE-COLUMN",
    "FULL_ENTRY"
]
final_df = final_df[cols]

# =====================
# SAVE
# =====================
final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print("✅ Done. Indexed CSV saved.")

In [86]:
# re-run failed rows, with RAW LOG file as backup
# =====================
# CONFIG
# =====================
MODEL = "gemma4:e2b"
BATCH_SIZE = 4

INPUT_FILE = "rundi_extracted_cleaned_full.txt"
INDEXED_CSV = "rundi_llm_tagged_WITH_INDEX.csv"
OUTPUT_CSV = "rundi_llm_tagged_FILLED.csv"
RAW_LOG_FILE = OUTPUT_CSV.replace(".csv", "_RAW_LOG.csv")

START_COLUMN = "P1-C1"

SYSTEM_PROMPT = """
You are performing STRICT information extraction.

You MUST extract data ONLY from the provided <entry> tags.

CRITICAL CONSTRAINTS:
- DO NOT invent, infer, or hallucinate any entries.
- DO NOT split a single entry into multiple outputs.
- DO NOT merge multiple entries.
- The number of output objects MUST EXACTLY match the number of <entry> inputs.
- Each output object MUST correspond to exactly one <entry> with the same id.

OUTPUT FORMAT:
Return ONLY valid JSON:
{
  "entries": [
    { "id": "0", ... },
    { "id": "1", ... }
  ]
}

FIELDS (per entry):
- id: copy from input <entry id='...'>
- WORD: Headword only
- AFFIX: Prefix/suffix with ONE hyphen (e.g., "-ye", "imy-") or null
- NOUN_CLASS: Numeric class (e.g., "9|10", "11") or null
- DESCRIPTION: Short French gloss ONLY (no examples)
- POS: Must be one of ["n.", "v.", "adj.", "adv.", "interj.", "prep.", "conj."]

STRICT RULES:
1. One input entry → one output object
2. Preserve order using id
3. If unsure → use null, NEVER invent
4. Do NOT output extra entries
5. Do NOT output text outside JSON
"""

# =====================
# LOAD INDEXED CSV
# =====================
df = pd.read_csv(INDEXED_CSV)

# =====================
# FIND MISSING INDICES
# =====================
missing_indices = df[df["WORD"].isna()]["GLOBAL_INDEX"].tolist()

# group into contiguous chunks
groups = []
current = []

for idx in sorted(missing_indices):
    if not current or idx == current[-1] + 1:
        current.append(idx)
    else:
        groups.append(current)
        current = [idx]

if current:
    groups.append(current)

# =====================
# SPLIT INTO BATCHES OF 4
# =====================
batches = []
for group in groups:
    for i in range(0, len(group), BATCH_SIZE):
        batches.append(group[i:i + BATCH_SIZE])

print(f"Total repair batches: {len(batches)}")

# =====================
# REBUILD processed_entries
# =====================
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    raw_chunks = [e.strip() for e in f.read().split('\n\n') if len(e.strip()) > 3]

processed_entries = []
current_column = "UNKNOWN"
start_processing = True if not START_COLUMN else False

for chunk in raw_chunks:
    if chunk.upper().startswith("COLUMN-START"):
        current_column = chunk
        if START_COLUMN and START_COLUMN in chunk:
            start_processing = True
    else:
        if start_processing:
            processed_entries.append((chunk, current_column))

# =====================
# REPAIR LOOP
# =====================

# reset raw log
open(RAW_LOG_FILE, 'w').close()
raw_header_written = False

# batches = batches[:3] # <<<<<-------- DEBUG

for batch in tqdm(batches, desc="Repairing Missing Rows"):

    batch_tuples = [processed_entries[i] for i in batch]

    batch_str = "\n".join([
        f"<entry id='{idx}'>{text}</entry>"
        for idx, (text, col) in enumerate(batch_tuples)
    ])
    # print(batch,'\n',batch_str,'\n\n')

    items = []
    failed_flag = 0

    for attempt in range(2):
        try:
            response = ollama.chat(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': f"Extract every entry within the tags into a JSON array labeled 'entries':\n{batch_str}"}
                ],
                format='json',
                options={
                    "temperature": 0,
                    "top_k": 1,
                    "top_p": 1.0, # ⚠️ IMPORTANT: disable nucleus truncation
                    "repeat_penalty": 1.1,
                    "num_ctx": 2048,
                    "num_predict": 1024  # reduce to avoid drift
                }
            )

            content = response['message']['content']
            raw_output = content  # always capture

            batch_data = json.loads(content)

            extracted_list = batch_data.get('entries', []) if isinstance(batch_data, dict) else batch_data
            if not isinstance(extracted_list, list):
                extracted_list = [extracted_list]

            output_map = {
                str(item.get('id')): item
                for item in extracted_list
                if isinstance(item, dict) and 'id' in item
            }

            batch_results = []

            for idx_local, global_idx in enumerate(batch):
                original_text, source_col = processed_entries[global_idx]

                item = output_map.get(str(idx_local))

                if item is None and len(extracted_list) == len(batch):
                    item = extracted_list[idx_local]

                if item:
                    item['PAGE-COLUMN'] = "-".join(source_col.split("-")[2:4])
                    item['FULL_ENTRY'] = original_text
                    item.pop('id', None)
                    item["GLOBAL_INDEX"] = global_idx
                    item["FAILED"] = 0
                    batch_results.append(item)

            if batch_results:
                items = batch_results
                break

        except Exception as e:
            if attempt == 1:
                print(f"❌ Failed batch {batch}: {e}")
            else:
                time.sleep(2)

    # =====================
    # FALLBACK
    # =====================
    if not items:
        failed_flag = 1
        for global_idx in batch:
            text, col = processed_entries[global_idx]
            items.append({
                "WORD": None,
                "AFFIX": None,
                "NOUN_CLASS": None,
                "DESCRIPTION": None,
                "POS": None,
                "PAGE-COLUMN": "-".join(col.split("-")[2:4]),
                "FULL_ENTRY": text,
                "GLOBAL_INDEX": global_idx,
                "FAILED": 1
            })

    # =====================
    # UPDATE DF
    # =====================
    for item in items:
        idx = item["GLOBAL_INDEX"]
        for col in ["WORD","AFFIX","NOUN_CLASS","DESCRIPTION","POS","FAILED"]:
            df.loc[df["GLOBAL_INDEX"] == idx, col] = item.get(col)

    # =====================
    # RAW LOG (APPEND)
    # =====================
    pd.DataFrame([{
        "BATCH_START_INDEX": batch[0],
        "BATCH_INDICES": batch,
        "FAILED": failed_flag,
        "RAW_OUTPUT": raw_output
    }]).to_csv(
        RAW_LOG_FILE,
        mode='a',
        index=False,
        header=not raw_header_written,
        encoding='utf-8'
    )

    raw_header_written = True

# =====================
# SAVE FINAL
# =====================
df = df.sort_values("GLOBAL_INDEX")
df.to_csv(OUTPUT_CSV, index=False)

print("✅ Missing rows repaired and saved.")

Total repair batches: 725


Repairing Missing Rows:   0%|          | 0/725 [00:00<?, ?it/s]

❌ Failed batch [1388, 1389, 1390, 1391]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [1532, 1533, 1534, 1535]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [1620, 1621, 1622, 1623]: Unterminated string starting at: line 1 column 495 (char 494)
❌ Failed batch [2184, 2185, 2186, 2187]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [2284, 2285, 2286, 2287]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [2528, 2529, 2530, 2531]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [2612, 2613, 2614, 2615]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [2728, 2729, 2730, 2731]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [2800, 2801, 2802, 2803]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [2836, 2837, 2838, 2839]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [2856, 2857, 2858, 2859]: Expecting value: line 1 column 1 (char 0)
❌ Failed batch [3040, 3041, 3042, 3043]: Expecting ',' delimiter: l

In [92]:
# batch size errors in the model outputs: so now use the raw log file to recover the outputs

import pandas as pd
import json
import re
import ast

# =====================
# CONFIG
# =====================
RAW_LOG_FILE = "rundi_llm_tagged_FILLED_RAW_LOG.csv"
INDEXED_CSV = "rundi_llm_tagged_WITH_INDEX.csv"
OUTPUT_CSV = "rundi_llm_tagged_RECOVERED.csv"


# =====================
# LOAD
# =====================
df_raw = pd.read_csv(RAW_LOG_FILE)
df_indexed = pd.read_csv(INDEXED_CSV)

# =====================
# SAFE JSON PARSER
# =====================
def safe_parse(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except:
                return None
    return None

# =====================
# NORMALIZE STRUCTURE
# =====================
def extract_entries(data):
    if isinstance(data, dict):
        return data.get("entries", [])
    elif isinstance(data, list):
        return data
    return []

# =====================
# MAIN LOGIC
# =====================
updates = 0

for _, row in df_raw.iterrows():

    raw_text = str(row["RAW_OUTPUT"])

    # 🔥 parse indices list safely
    try:
        batch_indices = ast.literal_eval(str(row["BATCH_INDICES"]))
    except:
        continue

    if not isinstance(batch_indices, list):
        continue

    parsed = safe_parse(raw_text)
    if not parsed:
        continue

    entries = extract_entries(parsed)

    # =====================
    # ALIGN ENTRIES → INDICES
    # =====================
    for idx_pos, global_idx in enumerate(batch_indices):

        if idx_pos >= len(entries):
            continue

        item = entries[idx_pos]

        if not isinstance(item, dict):
            continue

        # fetch original FULL_ENTRY
        match_row = df_indexed[df_indexed["GLOBAL_INDEX"] == global_idx]

        if match_row.empty:
            continue

        full_entry = str(match_row["FULL_ENTRY"].values[0]).strip()
        word = str(item.get("WORD", "")).strip()

        # 🔥 STRICT VALIDATION
        if not word or not full_entry.startswith(word):
            continue

        # =====================
        # UPDATE
        # =====================
        for col in ["WORD", "AFFIX", "NOUN_CLASS", "DESCRIPTION", "POS"]:
            df_indexed.loc[
                df_indexed["GLOBAL_INDEX"] == global_idx, col
            ] = item.get(col)

        updates += 1

print(f"✅ Updated {updates} rows")

# =====================
# SAVE
# =====================
df_indexed.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"✅ Saved to {OUTPUT_CSV}")

✅ Updated 1574 rows
✅ Saved to rundi_llm_tagged_RECOVERED.csv


In [136]:
# Run 3: fix further failing batches, one by one

import ollama
import pandas as pd
import json
import time
import re
from tqdm.notebook import tqdm

# =====================
# CONFIG
# =====================
MODEL = "gemma4:e2b"
BATCH_SIZE = 1

INDEXED_CSV = "rundi_llm_tagged_RECOVERED.csv"
OUTPUT_CSV = "rundi_llm_tagged_FINAL.csv"
RAW_LOG_FILE = OUTPUT_CSV.replace(".csv", "_RAW_LOG.csv")

# run 4: (FINAL, i hope)
INDEXED_CSV = "rundi_llm_tagged_FINAL.csv"
INPUT_RAW_LOG = "rundi_llm_tagged_FINAL_RAW_LOG_run3.csv"
OUTPUT_CSV = "rundi_llm_tagged_FINAL_run4.csv"
RAW_LOG_FILE = OUTPUT_CSV.replace(".csv", "_RAW_LOG.csv")

SYSTEM_PROMPT = """
Extract Rundi dictionary data into a JSON array labeled "entries".

STRICT RULES:
- Extract ONLY from the provided entries. DO NOT invent or split entries.
- The number of outputs MUST equal the number of <entry> inputs.
- Preserve order exactly.
- Each output corresponds to exactly one <entry id>.

For each <entry>, extract:
- WORD: Headword only (must match beginning of entry).
- AFFIX: Prefix/suffix with ONE hyphen (e.g., "-ye", "imy-"), else null.
- NOUN_CLASS: Numeric class (e.g., "9|10"), else null.
- DESCRIPTION: Short French gloss only.
- POS: Must always be present.

Return ONLY valid JSON:
{"entries": [...]}
"""

# =====================
# LOAD CSV
# =====================
df = pd.read_csv(INDEXED_CSV)

run3_log_df = pd.read_csv(INPUT_RAW_LOG)
# =====================
# FIND MISSING
# =====================
missing_indices = run3_log_df[run3_log_df["FAILED"]==1]["BATCH_START_INDEX"].tolist()

# group contiguous
groups = []
current = []

for idx in sorted(missing_indices):
    if not current or idx == current[-1] + 1:
        current.append(idx)
    else:
        groups.append(current)
        current = [idx]

if current:
    groups.append(current)

# split into batches of 4
batches = []
for group in groups:
    for i in range(0, len(group), BATCH_SIZE):
        batches.append(group[i:i + BATCH_SIZE])

print(f"Total repair batches: {len(batches)}")

# =====================
# RESET RAW LOG
# =====================
open(RAW_LOG_FILE, 'w').close()
raw_header_written = False

# =====================
# SAFE JSON PARSER
# =====================
def safe_parse(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except:
                return None
    return None

# =====================
# REPAIR LOOP
# =====================
for batch in tqdm(batches, desc="Repairing Missing Rows"):

    # 🔥 USE FULL_ENTRY DIRECTLY
    batch_entries = df[df["GLOBAL_INDEX"].isin(batch)].sort_values("GLOBAL_INDEX")

    batch_str = "\n".join([
        f"<entry id='{i}'>{row['FULL_ENTRY']}</entry>"
        for i, (_, row) in enumerate(batch_entries.iterrows())
    ])

    items = []
    raw_output = ""
    failed_flag = 0

    for attempt in range(3):
        try:
            response = ollama.chat(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': f"Extract entries:\n{batch_str}"}
                ],
                options={
                    "temperature": 0,
                    "top_k": 1,
                    "top_p": 1.0,
                    "repeat_penalty": 1.15,
                    "num_ctx": 2048,
                    "num_predict": 1024
                }
            )

            raw_output = response['message'].get('content', '')

            if not raw_output.strip():
                time.sleep(1)
                continue

            parsed = safe_parse(raw_output)
            if not parsed:
                continue

            extracted_list = parsed.get("entries", []) if isinstance(parsed, dict) else parsed
            if not isinstance(extracted_list, list):
                extracted_list = [extracted_list]

            batch_results = []

            for idx_local, (_, row) in enumerate(batch_entries.iterrows()):
                global_idx = row["GLOBAL_INDEX"]
                full_entry = str(row["FULL_ENTRY"]).strip()

                item = extracted_list[idx_local] if idx_local < len(extracted_list) else None

                if not isinstance(item, dict):
                    continue

                word = str(item.get("WORD", "")).strip()

                # 🔥 ANTI-HALLUCINATION CHECK
                if not word or not full_entry.startswith(word):
                    continue

                item["GLOBAL_INDEX"] = global_idx
                item["FAILED"] = 0

                batch_results.append(item)

            if batch_results:
                items = batch_results
                break

        except Exception as e:
            if attempt == 2:
                print(f"❌ Failed batch {batch}: {e}")
            else:
                time.sleep(1)

    # =====================
    # FALLBACK
    # =====================
    if not items:
        failed_flag = 1
        for _, row in batch_entries.iterrows():
            items.append({
                "WORD": None,
                "AFFIX": None,
                "NOUN_CLASS": None,
                "DESCRIPTION": None,
                "POS": None,
                "GLOBAL_INDEX": row["GLOBAL_INDEX"],
                "FAILED": 1
            })

    # =====================
    # UPDATE DF (IN PLACE)
    # =====================
    for item in items:
        idx = item["GLOBAL_INDEX"]
        for col in ["WORD","AFFIX","NOUN_CLASS","DESCRIPTION","POS","FAILED"]:
            df.loc[df["GLOBAL_INDEX"] == idx, col] = item.get(col)

    # =====================
    # RAW LOG
    # =====================
    pd.DataFrame([{
        "BATCH_START_INDEX": batch[0],
        "BATCH_INDICES": batch,
        "FAILED": failed_flag,
        "RAW_OUTPUT": raw_output
    }]).to_csv(
        RAW_LOG_FILE,
        mode='a',
        index=False,
        header=not raw_header_written,
        encoding='utf-8'
    )

    raw_header_written = True

# =====================
# SAVE FINAL
# =====================
df = df.sort_values("GLOBAL_INDEX")
df.to_csv(OUTPUT_CSV, index=False)

print("✅ Missing rows repaired and saved.")

Total repair batches: 222


Repairing Missing Rows:   0%|          | 0/222 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [89]:
df_new = df_new.dropna(subset=['WORD'])

# df_new.to_csv('rundi_llm_tagged_gemma4_fromP1-C1_FIXED.csv',index=False)

In [55]:
# batch 371 * 4 = 1484

df_fixed[df_fixed['GLOBAL_INDEX'] == 1484]

,WORD,AFFIX,NOUN_CLASS,DESCRIPTION,POS,PAGE-COLUMN,FULL_ENTRY,GLOBAL_INDEX
1484,ibundi,ama-,5'6,"tas de terre, butte.",n.,P45-C1,"ibundi, ama-, 5'6, (cfr -bûndik-), tas de terr...",1484


Total entries: 19415
✅ Done. Indexed CSV saved.


JSONDecodeError: Expecting ',' delimiter: line 84 column 6 (char 1776)

In [50]:
df = pd.read_csv('rundi_llm_sample.csv', na_filter=True, usecols=['WORD','AFFIX','NOUN_CLASS','DESCRIPTION','POS','PAGE-COLUMN','FULL_ENTRY'])
df.head()

,WORD,AFFIX,NOUN_CLASS,DESCRIPTION,POS,PAGE-COLUMN,FULL_ENTRY
0,a,NaN,NaN,"interjection de négation, de dénégation: non.",interj.,P1-C1,"a, (inv.), interjection de négation, de dénéga..."
1,umwaba,imy-,34,surface de terre arable dans le marais.,n.,P1-C1,"umwaba, imy-, 34, surface de terre arable dans..."
2,akāba,utw-,12|13,lopin de terre arable. Drain dans les champs d...,n.,P1-C1,"akāba, utw-, 12|13, lopin de terre arable. Dra..."
3,kwâba,-âvye,NaN,"crier vers; avoir recours à, recourir à.",v.,P1-C1,"kwâba, -âvye, crier vers; avoir recours à, rec..."
4,kwabāba,-vye,NaN,Tendre la main pour prendre en se dressant sur...,v.,P1-C1,"kwabāba, -vye, (-ab--), tendre la main pour pr..."


In [58]:
df.describe()

,WORD,AFFIX,NOUN_CLASS,DESCRIPTION,POS,PAGE-COLUMN,FULL_ENTRY
count,11272,8881,4863,10907,11258,11272,11272
unique,10995,1560,292,10387,33,757,11272
top,impeta,-ye,9|10,bonne famille hutu.,n.,P33-C2,"a, (inv.), interjection de négation, de dénéga..."
freq,3,1322,646,12,5446,33,1


In [49]:
# train a text classification model with cosine similarity?

In [51]:
# step 3
df_nc = df.dropna(subset = ['NOUN_CLASS'])

In [52]:
df_nc[:52]

,WORD,AFFIX,NOUN_CLASS,DESCRIPTION,POS,PAGE-COLUMN,FULL_ENTRY
1,umwaba,imy-,34,surface de terre arable dans le marais.,n.,P1-C1,"umwaba, imy-, 34, surface de terre arable dans..."
2,akāba,utw-,12|13,lopin de terre arable. Drain dans les champs d...,n.,P1-C1,"akāba, utw-, 12|13, lopin de terre arable. Dra..."
6,umwâbagirano,imy-,3|4,"Beuglement, appels du bétail au pâturage.",n.,P1-C1,"umwâbagirano, imy-, 3|4, (cfr -âbagir-), beugl..."
7,urwăbagiza,(cfr -âbagiz-),11,Philtre amouYEUX.,n.,P1-C1,"urwăbagiza, 11, (cfr -âbagiz-), philtre amouYEUX,"
13,inyawïyabira,bānyáwīïyabira,9lal2a,"femme qui a habitude de gémir, hypocondre",n.,P1-C1,"inyawïyabira, bānyáwīïyabira, 9lal2a, (cfr -iy..."
20,inyăbu,NaN,9|10,égratignure,n.,P1-C2,"inyăbu, 9|10, (cfr -abur-), égratignure."
21,kwabura,-ye,9,égratigner,v.,P1-C2,"kwabura, -ye, (cfr -abu 9), égratigner."
23,umwabūzo,imy-,34,balai,n.,P1-C2,"umwabūzo, imy-, 34, (cfr -abūr-), balai. Syn. ..."
25,icāduka,ivy-,7|8,accident,n.,P1-C2,"icāduka, ivy-, 7|8, (cfr -âduk-), accident, év..."
26,kwâduka,-tse,7,venir de,v.,P1-C2,"kwâduka, -tse, venir de, provenir de, gikôkó c..."
